# 13 · Reinforcement Learning

**Part VI — Reinforcement learning** · Estimated time: ~120 min

## Learning objectives

- Formalize sequential decision making as a Markov decision process and derive the Bellman expectation and optimality equations.
- Use the Gymnasium environment API (`reset`, `step`, observation and action spaces) and measure a random-agent baseline.
- Implement value iteration, Q-learning and SARSA on FrozenLake and visualize the learned policies.
- Derive the policy gradient theorem and implement REINFORCE with reward-to-go and a learned baseline.
- Implement Deep Q-Networks with experience replay, a target network and the Huber loss, and improve them with Double DQN.
- Implement an advantage actor-critic with an entropy bonus and generalized advantage estimation, and a compact PPO with the clipped objective.
- Compare the sample efficiency and stability of the different algorithm families on CartPole.
- Explain how reinforcement learning is used in practice, including reinforcement learning from human feedback for language models.

## Prerequisites

- Notebook 01 · Mathematical Foundations (expectations, conditional probability, gradients).
- Notebook 03 · Tensors and Autograd (autograd, `torch.no_grad`, gradients of losses that contain sampled quantities).
- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (Adam, gradient clipping, learning-rate schedules).
- Notebook 05 · The PyTorch Workflow and Best Practices (`nn.Module`, training loops, seeding).

## Contents

1. The reinforcement learning problem
2. The Gymnasium API and a random baseline
3. Tabular methods on FrozenLake: value iteration, Q-learning, SARSA
4. Policy gradients: REINFORCE with reward-to-go and a baseline
5. Deep Q-Networks and Double DQN
6. Actor-critic methods: A2C, generalized advantage estimation and PPO
7. Reinforcement learning in the wild, and RLHF

In [ ]:
import copy, math, os, random, time
from collections import deque, namedtuple
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# gymnasium is an optional dependency: if it is missing we fall back to small in-notebook
# re-implementations of CartPole and FrozenLake (defined in Section 2) with the same interface.
try:
    import gymnasium as gym
    GYM_AVAILABLE = True
    print(f"gymnasium {gym.__version__} available")
except ImportError:
    gym = None
    GYM_AVAILABLE = False
    print("gymnasium is not installed: using the minimal in-notebook environments instead.\n"
          "Create the course environment from environment.yml to use the real Gymnasium environments.")

## 1. The reinforcement learning problem

Every notebook so far has been about **supervised** or **unsupervised** learning: a dataset is handed to us, the loss is a fixed function of the model output and a label (or of the input itself), and the only thing the model influences is its own prediction. **Reinforcement learning** (RL) breaks that separation. The learner, called the **agent**, *acts* in an **environment**; each action changes the situation the agent finds itself in and produces a scalar **reward**. The agent's goal is not to reproduce a label but to choose actions, over time, so that the total reward it collects is as large as possible. Because the agent's choices determine which situations it will encounter next, and therefore which data it will learn from, RL is a problem of *learning by interaction*.

### 1.1 Agents, environments and the interaction loop

The interaction is a loop. At time step $t$ the agent observes a **state** $S_t$ (in practice an *observation*, which may or may not reveal the full state), chooses an **action** $A_t$, and the environment responds with a reward $R_{t+1}$ and a new state $S_{t+1}$. The loop continues until a **terminal** state is reached, which closes one **episode**; then the environment is reset and a new episode starts. Some tasks are naturally episodic (a game of chess, a robot arm reaching for a cup), others are continuing (a thermostat), but in this notebook every task has episodes.

Three features of this loop make RL genuinely different from the learning we have done so far.

1. **The feedback is evaluative, not instructive.** A supervised label says "the right answer was 7". A reward says "that was worth 0.3", without saying what would have been better. The agent has to discover good actions by trying them.
2. **Rewards can be delayed.** The move that lost a chess game may have been played twenty moves before checkmate. Assigning credit to the actions that actually caused a late reward is the **credit assignment problem**.
3. **The data distribution depends on the policy.** A poorly-driving agent only ever sees the states a bad driver sees. Improving the policy changes the training data, which makes the optimization non-stationary and is the root of most of the instabilities we will meet in Section 5.

> **Key idea.** Reinforcement learning is the problem of choosing actions to maximize cumulative reward when the consequences of actions are delayed and the learner's own choices determine what it gets to observe.

### 1.2 Markov decision processes

To reason precisely we need a mathematical model of the environment. A **Markov decision process** (MDP) is a tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$ where

- $\mathcal{S}$ is the set of states and $\mathcal{A}$ the set of actions (both finite in our tabular examples; $\mathcal{S}$ is a continuous vector space for CartPole);
- $P(s' \mid s, a)$ is the **transition kernel**: the probability of landing in state $s'$ after taking action $a$ in state $s$;
- $R(s, a, s')$ is the **reward function**, the expected immediate reward for that transition (we will write $r$ for a realized reward);
- $\gamma \in [0, 1]$ is the **discount factor**, explained in the next subsection.

The defining property is the **Markov property**: the next state and reward depend only on the current state and action, not on the earlier history,

$$
\Pr(S_{t+1} = s', R_{t+1} = r \mid S_t, A_t, S_{t-1}, A_{t-1}, \dots, S_0, A_0) = \Pr(S_{t+1} = s', R_{t+1} = r \mid S_t, A_t).
$$

The state is, by definition, a sufficient statistic of the past. This is what will let us write value functions of the *current state only*, which is the key to every algorithm in this notebook. When the observation is not Markov (a single video frame does not reveal velocities), practitioners stack several frames or use a recurrent network so that the network input becomes approximately Markov again.

The agent's behaviour is described by a **policy** $\pi(a \mid s)$, a probability distribution over actions for every state. A deterministic policy is the special case where all the mass sits on one action, $\pi(s) = a$. In the deep RL sections the policy will be a neural network that maps an observation to the logits of a categorical distribution over actions.

### 1.3 Returns and discounting

The agent wants a lot of reward *in total*, not just now. The quantity it maximizes is the **return** from time $t$, the discounted sum of all future rewards:

$$
G_t \;=\; R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots \;=\; \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}.
$$

The discount factor $\gamma$ plays three roles at once. Mathematically it keeps the infinite sum finite: if rewards are bounded by $r_{\max}$ then $|G_t| \le r_{\max} / (1 - \gamma)$. Economically it expresses that a reward now is worth more than the same reward later (an interest rate). Statistically it limits how far into the future the agent must look, which reduces the variance of everything we estimate: rewards more than roughly $1/(1-\gamma)$ steps away barely matter. This **effective horizon** $1/(1-\gamma)$ is the number to keep in mind: $\gamma = 0.99$ means the agent cares about roughly the next hundred steps, $\gamma = 0.9$ about the next ten.

The return satisfies a one-step recursion that we will use everywhere:

$$
G_t = R_{t+1} + \gamma G_{t+1}.
$$

Writing $G_t = R_{t+1} + \gamma (R_{t+2} + \gamma R_{t+3} + \dots)$ makes the recursion obvious, and it also gives an $O(T)$ algorithm to compute all returns of an episode: walk backwards from the last reward. Let us implement that and look at how $\gamma$ shapes the horizon.

In [ ]:
def discounted_returns(rewards, gamma):
    """Return the array G with G[t] = sum_{k>=0} gamma^k * rewards[t+k], computed backwards in O(T)."""
    G = np.zeros(len(rewards), dtype=np.float64)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running     # G_t = R_{t+1} + gamma * G_{t+1}
        G[t] = running
    return G

# An episode with reward 1 at every step, as in CartPole. G_0 saturates at 1 / (1 - gamma).
horizons = np.arange(1, 201)
plt.figure()
for gamma in [0.5, 0.9, 0.99]:
    G0 = [discounted_returns(np.ones(T), gamma)[0] for T in horizons]
    plt.plot(horizons, G0, label=f"gamma = {gamma}  (1/(1-gamma) = {1 / (1 - gamma):.0f})")
plt.xlabel("episode length T (reward 1 at every step)")
plt.ylabel("discounted return G_0")
plt.title("The discount factor sets an effective horizon")
plt.legend(); plt.tight_layout(); plt.show()

print("Returns of a 6-step episode with rewards [0, 0, 0, 0, 0, 1] and gamma = 0.9:")
print(np.round(discounted_returns(np.array([0, 0, 0, 0, 0, 1.0]), 0.9), 3))

With $\gamma = 0.5$ the return of an episode is essentially determined by its first three or four rewards; with $\gamma = 0.99$ an episode of 200 steps still has not saturated, so the agent is rewarded for surviving longer. The second print shows credit assignment in miniature: a single reward at the end of the episode is "propagated" backwards by the recursion, and each earlier step receives a share that shrinks geometrically with its distance from the reward. All the algorithms below are, in one way or another, machines for propagating rewards backwards through time.

### 1.4 Policies and value functions

Because the environment and the policy are both stochastic, the return of a state is a random variable. The **state-value function** of a policy is its expectation,

$$
V^\pi(s) \;=\; \mathbb{E}_\pi\!\left[ G_t \mid S_t = s \right],
$$

where $\mathbb{E}_\pi$ means that actions are drawn from $\pi$ and transitions from $P$. The **action-value function** additionally fixes the first action,

$$
Q^\pi(s, a) \;=\; \mathbb{E}_\pi\!\left[ G_t \mid S_t = s, A_t = a \right].
$$

The two are related by averaging over the policy: $V^\pi(s) = \sum_a \pi(a \mid s)\, Q^\pi(s, a)$. Intuitively $V^\pi(s)$ answers "how good is it to be here if I keep behaving like $\pi$?" and $Q^\pi(s, a)$ answers "how good is it to do $a$ here and then keep behaving like $\pi$?". The second question is the more useful one for *improving* a policy: if $Q^\pi(s, a') > V^\pi(s)$ for some action $a'$, then switching to $a'$ in state $s$ is an improvement. Value functions are therefore both a diagnostic and the engine of learning.

An **optimal policy** $\pi^*$ is one whose value is at least as large as that of any other policy in every state. For finite MDPs such a policy always exists and it can be chosen deterministic; its value functions are written $V^*$ and $Q^*$, and $\pi^*(s) = \arg\max_a Q^*(s, a)$. Knowing $Q^*$ is therefore the same as knowing how to act optimally, which is why so much of RL is about estimating $Q^*$.

### 1.5 The Bellman expectation equations

The recursion $G_t = R_{t+1} + \gamma G_{t+1}$ for the random return turns into a recursion for the value function. Take the conditional expectation of both sides given $S_t = s$:

$$
V^\pi(s) = \mathbb{E}_\pi\!\left[ R_{t+1} \mid S_t = s \right] + \gamma\, \mathbb{E}_\pi\!\left[ G_{t+1} \mid S_t = s \right].
$$

For the second term we use the tower property of expectations, conditioning additionally on $S_{t+1}$, together with the Markov property (the return from $t+1$ on depends on the past only through $S_{t+1}$): $\mathbb{E}_\pi[G_{t+1} \mid S_t = s] = \mathbb{E}_\pi\big[ \mathbb{E}_\pi[G_{t+1} \mid S_{t+1}] \mid S_t = s \big] = \mathbb{E}_\pi[V^\pi(S_{t+1}) \mid S_t = s]$. Expanding the expectations over the action and the next state gives the **Bellman expectation equation** for $V^\pi$:

$$
V^\pi(s) \;=\; \sum_{a} \pi(a \mid s) \sum_{s'} P(s' \mid s, a)\, \big[ R(s, a, s') + \gamma\, V^\pi(s') \big].
$$

Exactly the same argument with the first action held fixed gives the equation for $Q^\pi$:

$$
Q^\pi(s, a) \;=\; \sum_{s'} P(s' \mid s, a)\, \Big[ R(s, a, s') + \gamma \sum_{a'} \pi(a' \mid s')\, Q^\pi(s', a') \Big].
$$

These are *linear* equations in the unknowns $V^\pi(s)$, one per state. Collecting them into vectors, with $P_\pi(s, s') = \sum_a \pi(a \mid s) P(s' \mid s, a)$ the state-to-state transition matrix under $\pi$ and $r_\pi(s)$ the expected one-step reward, they read $V^\pi = r_\pi + \gamma P_\pi V^\pi$, whose solution is

$$
V^\pi = (I - \gamma P_\pi)^{-1} r_\pi .
$$

The inverse exists because $P_\pi$ is a stochastic matrix (its spectral radius is 1) and $\gamma < 1$. Solving this system is called **policy evaluation**. For a handful of states we can literally call a linear solver; for large or continuous state spaces we cannot, and the rest of the notebook is about doing policy evaluation and improvement *from samples* instead.

There is a second way to solve the system that generalizes far better than matrix inversion: treat the right-hand side as an operator $\mathcal{T}^\pi V = r_\pi + \gamma P_\pi V$ and iterate $V_{k+1} = \mathcal{T}^\pi V_k$ from any starting guess. Because $\|P_\pi u\|_\infty \le \|u\|_\infty$ for a stochastic matrix, the operator is a **$\gamma$-contraction** in the max norm,

$$
\|\mathcal{T}^\pi V - \mathcal{T}^\pi U\|_\infty = \gamma \|P_\pi (V - U)\|_\infty \le \gamma \|V - U\|_\infty ,
$$

so by the Banach fixed-point theorem the iteration converges to the unique fixed point $V^\pi$, and the error shrinks by at least a factor $\gamma$ per sweep. Let us verify both facts on a three-state MDP small enough to write down by hand.

In [ ]:
# A three-state MDP. Action 0 = "stay" (nothing happens, no reward). Action 1 = "move": advances
# 0 -> 1 -> 2 with probability 0.8 (and stays put otherwise); from state 2 "move" pays reward 1 and
# returns to state 0 with certainty. P[a, s, s'] is the transition probability, R[a, s] the expected reward.
n_states, n_actions, gamma_demo = 3, 2, 0.9
P_demo = np.zeros((n_actions, n_states, n_states))
R_demo = np.zeros((n_actions, n_states))
P_demo[0] = np.eye(n_states)
P_demo[1] = np.array([[0.2, 0.8, 0.0],
                      [0.0, 0.2, 0.8],
                      [1.0, 0.0, 0.0]])
R_demo[1, 2] = 1.0

def policy_matrices(pi, P, R):
    """State-to-state transition matrix P_pi and expected reward vector r_pi for a policy pi[s, a]."""
    P_pi = np.einsum("sa,ask->sk", pi, P)
    r_pi = np.einsum("sa,as->s", pi, R)
    return P_pi, r_pi

def evaluate_policy_exact(pi, P, R, gamma):
    """Solve the Bellman expectation equation V = r_pi + gamma P_pi V as a linear system."""
    P_pi, r_pi = policy_matrices(pi, P, R)
    return np.linalg.solve(np.eye(len(r_pi)) - gamma * P_pi, r_pi)

def evaluate_policy_iterative(pi, P, R, gamma, n_sweeps=50):
    """Iterate the Bellman operator V <- r_pi + gamma P_pi V and record the error to the exact solution."""
    P_pi, r_pi = policy_matrices(pi, P, R)
    V_exact = evaluate_policy_exact(pi, P, R, gamma)
    V, errors = np.zeros(len(r_pi)), []
    for _ in range(n_sweeps):
        V = r_pi + gamma * P_pi @ V
        errors.append(np.abs(V - V_exact).max())
    return V, np.array(errors)

pi_move = np.array([[0.0, 1.0]] * n_states)   # always "move"
pi_coin = np.array([[0.5, 0.5]] * n_states)   # coin flip in every state
plt.figure()
for name, pi in [("always move", pi_move), ("coin flip", pi_coin)]:
    V_exact = evaluate_policy_exact(pi, P_demo, R_demo, gamma_demo)
    V_iter, errors = evaluate_policy_iterative(pi, P_demo, R_demo, gamma_demo)
    print(f"{name:12s}: V^pi = {np.round(V_exact, 4)}   max error after {len(errors)} sweeps = {errors[-1]:.1e}")
    plt.semilogy(np.arange(1, len(errors) + 1), errors, label=f"{name}: ||V_k - V^pi||")
plt.semilogy(np.arange(1, 51), errors[0] * gamma_demo ** np.arange(50), "k--", label="gamma^k (contraction bound)")
plt.xlabel("sweep k"); plt.ylabel("max-norm error"); plt.title("Iterating the Bellman operator converges geometrically")
plt.legend(); plt.tight_layout(); plt.show()

The linear solve and the fixed-point iteration agree to numerical precision, and the error decays at the rate predicted by the contraction argument (the dashed line is $\gamma^k$; the actual decay is faster because the bound is not tight for these particular matrices). The values also make sense: under "always move", state 2 is worth the most because the reward is one step away, state 1 slightly less, state 0 the least; the coin-flip policy wastes half its steps standing still and every value drops.

### 1.6 The Bellman optimality equations

The optimal value functions satisfy a similar recursion, but with the policy's average replaced by a *maximum* over actions, because an optimal agent takes the best action in every state:

$$
V^*(s) = \max_a \sum_{s'} P(s' \mid s, a)\, \big[ R(s, a, s') + \gamma V^*(s') \big],
\qquad
Q^*(s, a) = \sum_{s'} P(s' \mid s, a)\, \Big[ R(s, a, s') + \gamma \max_{a'} Q^*(s', a') \Big].
$$

The maximum makes these equations non-linear, so there is no closed-form solve. But the optimality operator $\mathcal{T}^* V = \max_a [\,r_a + \gamma P_a V\,]$ is *still* a $\gamma$-contraction: for any two vectors, $|\max_a x_a - \max_a y_a| \le \max_a |x_a - y_a|$, so the same argument as before goes through. Iterating $V_{k+1} = \mathcal{T}^* V_k$ is called **value iteration**; it converges to $V^*$ from any start, and reading off the greedy policy $\pi(s) = \arg\max_a Q(s, a)$ from the converged $Q$ gives an optimal policy. We implement it in Section 3 on FrozenLake, where the transition table is available.

Two remarks connect this to what comes later. First, *every* algorithm in this notebook is a way of solving a Bellman equation: value iteration solves it exactly with a known model; Q-learning solves the optimality equation from samples; DQN solves it from samples with a neural network standing in for the table; actor-critic methods solve the expectation equation for the current policy while simultaneously improving that policy. Second, the contraction property is what makes the tabular methods provably convergent, and it is precisely what *breaks* once we add function approximation and off-policy sampling, a problem known as the deadly triad (Section 5.1).

### 1.7 Exploration versus exploitation

A learner that only ever takes the action it currently believes is best will never find out whether another action is better. This is the **exploration–exploitation trade-off**: exploiting current knowledge gives reward now, exploring gathers information that may give more reward later. Supervised learning has no such problem, because the dataset is fixed. In RL the trade-off is unavoidable, and the simplest resolution, used throughout this notebook, is the **$\varepsilon$-greedy** policy: with probability $\varepsilon$ take a uniformly random action, otherwise take the greedy one. Typically $\varepsilon$ starts near 1 and is annealed towards a small value such as 0.05, so that early training explores broadly and late training refines. Policy-gradient methods explore differently: their policy is stochastic by construction, and an **entropy bonus** (Section 6) discourages it from collapsing onto one action too early.

### 1.8 How RL differs from supervised learning

It is worth stating the differences in one place, because they explain why the training loops in this notebook look unlike anything in Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization and Notebook 05 · The PyTorch Workflow and Best Practices.

| | Supervised learning | Reinforcement learning |
|---|---|---|
| Data | fixed i.i.d. dataset | generated by the agent, correlated in time, distribution shifts as the policy improves |
| Feedback | correct label for each input | scalar reward, possibly delayed, for the whole trajectory |
| Objective | minimize a loss the model can differentiate directly | maximize expected return; the environment is a black box we cannot differentiate through |
| Evaluation | held-out accuracy | average return of the policy, which has high variance and depends on the environment's randomness |
| Typical failure | overfitting | instability, high variance, reward hacking, insufficient exploration |

The practical consequences are that learning curves are noisy (we always smooth them with a moving average), that results vary a lot between random seeds (serious papers report several seeds), and that hyperparameters such as the discount factor and the exploration schedule matter as much as the learning rate.

## 2. The Gymnasium API and a random baseline

**Gymnasium** (the maintained fork of OpenAI Gym) is the standard Python interface to RL environments. An environment object exposes two spaces and two methods:

- `env.observation_space` and `env.action_space` describe the shapes and ranges of observations and actions. CartPole has a `Box(4,)` observation (cart position, cart velocity, pole angle, pole angular velocity) and a `Discrete(2)` action (push left or right). FrozenLake has `Discrete(16)` observations and `Discrete(4)` actions.
- `obs, info = env.reset(seed=...)` starts a new episode and returns the first observation.
- `obs, reward, terminated, truncated, info = env.step(action)` applies an action. **`terminated`** is true when the MDP reached a terminal state (the pole fell, the agent fell into a hole). **`truncated`** is true when the episode was cut off by a *time limit* that is not part of the MDP (CartPole is truncated after 500 steps, FrozenLake after 100). The distinction matters for bootstrapping: after termination the future return is exactly zero, but after truncation it is not, and treating the two the same introduces a subtle bias that we will handle explicitly in Sections 5 and 6.

Because `gymnasium` is an optional dependency, the next cell defines two tiny environments with the same interface as a fallback: `SimpleCartPole` implements the classic cart-pole physics of Barto, Sutton and Anderson (1983) exactly as Gymnasium does (Euler integration with a 0.02 s time step, termination at $|x| > 2.4$ or $|\theta| > 12^\circ$, truncation at 500 steps), and `SimpleFrozenLake` builds the same transition table `P` that Gymnasium's FrozenLake exposes, so that Section 3 can read the model from either implementation. The `make_env` helper picks whichever is available; everything downstream only uses the shared interface.

In [ ]:
class DiscreteSpace:
    """Minimal stand-in for gymnasium.spaces.Discrete (used only if gymnasium is missing)."""
    def __init__(self, n, seed=None):
        self.n = int(n)
        self._rng = np.random.default_rng(seed)
    def sample(self):
        return int(self._rng.integers(self.n))
    def seed(self, seed=None):
        self._rng = np.random.default_rng(seed)
    def __repr__(self):
        return f"Discrete({self.n})"


class BoxSpace:
    """Minimal stand-in for gymnasium.spaces.Box (only .shape, .low, .high and .sample())."""
    def __init__(self, low, high, seed=None):
        self.low, self.high = np.asarray(low, np.float32), np.asarray(high, np.float32)
        self.shape = self.low.shape
        self._rng = np.random.default_rng(seed)
    def sample(self):
        return self._rng.uniform(-1.0, 1.0, size=self.shape).astype(np.float32)
    def seed(self, seed=None):
        self._rng = np.random.default_rng(seed)
    def __repr__(self):
        return f"Box(shape={self.shape}, dtype=float32)"


class SimpleCartPole:
    """The classic cart-pole (Barto, Sutton & Anderson, 1983) with the Gymnasium 1.x interface.

    Observation: [x, x_dot, theta, theta_dot]; actions: 0 = push left, 1 = push right; reward 1 per step.
    The episode terminates when |x| > 2.4 or |theta| > 12 degrees, and is truncated after 500 steps.
    """
    gravity, masscart, masspole, length = 9.8, 1.0, 0.1, 0.5   # length = half the pole length
    force_mag, tau = 10.0, 0.02
    theta_threshold, x_threshold = 12 * 2 * math.pi / 360, 2.4
    max_episode_steps = 500

    def __init__(self):
        high = np.array([self.x_threshold * 2, np.finfo(np.float32).max,
                         self.theta_threshold * 2, np.finfo(np.float32).max], dtype=np.float32)
        self.observation_space = BoxSpace(-high, high)
        self.action_space = DiscreteSpace(2)
        self.rng = np.random.default_rng()
        self.state, self.t = None, 0

    @property
    def unwrapped(self):
        return self

    def reset(self, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.state = self.rng.uniform(-0.05, 0.05, size=4)
        self.t = 0
        return self.state.astype(np.float32), {}

    def step(self, action):
        x, x_dot, theta, theta_dot = self.state
        force = self.force_mag if int(action) == 1 else -self.force_mag
        total_mass = self.masscart + self.masspole
        polemass_length = self.masspole * self.length
        cos_t, sin_t = math.cos(theta), math.sin(theta)
        temp = (force + polemass_length * theta_dot ** 2 * sin_t) / total_mass
        theta_acc = (self.gravity * sin_t - cos_t * temp) / (
            self.length * (4.0 / 3.0 - self.masspole * cos_t ** 2 / total_mass))
        x_acc = temp - polemass_length * theta_acc * cos_t / total_mass
        # Explicit Euler integration, exactly as in Gymnasium's CartPole-v1.
        x, x_dot = x + self.tau * x_dot, x_dot + self.tau * x_acc
        theta, theta_dot = theta + self.tau * theta_dot, theta_dot + self.tau * theta_acc
        self.state = np.array([x, x_dot, theta, theta_dot])
        self.t += 1
        terminated = bool(abs(x) > self.x_threshold or abs(theta) > self.theta_threshold)
        truncated = bool(self.t >= self.max_episode_steps and not terminated)
        return self.state.astype(np.float32), 1.0, terminated, truncated, {}

    def close(self):
        pass


FROZEN_LAKE_4x4 = ["SFFF",
                   "FHFH",
                   "FFFH",
                   "HFFG"]

class SimpleFrozenLake:
    """4x4 FrozenLake with the Gymnasium interface and the same transition table `P[s][a]`.

    Each entry of P[s][a] is a list of (probability, next_state, reward, terminated). Actions are
    0 = left, 1 = down, 2 = right, 3 = up. On slippery ice the intended direction is taken with
    probability 1/3 and each of the two perpendicular directions with probability 1/3.
    """
    def __init__(self, is_slippery=True, max_episode_steps=100):
        self.desc = np.array([list(row) for row in FROZEN_LAKE_4x4], dtype="c")
        self.nrow, self.ncol = self.desc.shape
        self.nS, self.nA = self.nrow * self.ncol, 4
        self.observation_space = DiscreteSpace(self.nS)
        self.action_space = DiscreteSpace(self.nA)
        self.max_episode_steps = max_episode_steps
        self.rng = np.random.default_rng()
        self.s, self.t = 0, 0
        self.P = {s: {a: [] for a in range(self.nA)} for s in range(self.nS)}
        for r in range(self.nrow):
            for c in range(self.ncol):
                s = r * self.ncol + c
                letter = self.desc[r, c].decode()
                for a in range(self.nA):
                    if letter in "GH":                       # terminal states are absorbing
                        self.P[s][a].append((1.0, s, 0.0, True))
                        continue
                    moves = [(a - 1) % 4, a, (a + 1) % 4] if is_slippery else [a]
                    for b in moves:
                        r2, c2 = self._move(r, c, b)
                        letter2 = self.desc[r2, c2].decode()
                        self.P[s][a].append((1.0 / len(moves), r2 * self.ncol + c2,
                                             float(letter2 == "G"), letter2 in "GH"))

    def _move(self, r, c, a):
        if a == 0: c = max(c - 1, 0)
        elif a == 1: r = min(r + 1, self.nrow - 1)
        elif a == 2: c = min(c + 1, self.ncol - 1)
        elif a == 3: r = max(r - 1, 0)
        return r, c

    @property
    def unwrapped(self):
        return self

    def reset(self, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.s, self.t = 0, 0
        return 0, {"prob": 1.0}

    def step(self, action):
        transitions = self.P[self.s][int(action)]
        i = self.rng.choice(len(transitions), p=[tr[0] for tr in transitions])
        prob, s2, reward, terminated = transitions[i]
        self.s, self.t = s2, self.t + 1
        truncated = bool(self.t >= self.max_episode_steps and not terminated)
        return int(s2), float(reward), bool(terminated), truncated, {"prob": prob}

    def close(self):
        pass


def make_env(name, **kwargs):
    """Create an environment by name: Gymnasium if available, otherwise the in-notebook fallback."""
    if GYM_AVAILABLE:
        return gym.make(name, **kwargs)
    if name.startswith("CartPole"):
        return SimpleCartPole()
    if name.startswith("FrozenLake"):
        return SimpleFrozenLake(**kwargs)
    raise ValueError(f"No fallback implementation for {name}")

env = make_env("CartPole-v1")
print("using", type(env).__name__ if not GYM_AVAILABLE else f"gymnasium {env.spec.id}")

### 2.1 One episode step by step

Let us look at the raw interface once, because every training loop later is built on these few lines. We seed the *first* `reset` so that the notebook is reproducible; subsequent resets continue the environment's own random stream, which is the recommended Gymnasium pattern (seeding every reset with the same value would make every episode identical).

In [ ]:
obs, info = env.reset(seed=SEED)
print("observation space:", env.observation_space)
print("action space     :", env.action_space)
print("first observation:", np.round(obs, 4), "| dtype", obs.dtype, "| shape", obs.shape)
for t in range(4):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"t={t}: action={action}  reward={reward:.0f}  terminated={terminated}  truncated={truncated}  "
          f"obs={np.round(obs, 3)}")

The observation is a float32 vector of four physical quantities; the reward is 1 for every step in which the pole is still up, so the return of an episode equals its length and the maximum return is 500. The `info` dictionary is empty for CartPole; FrozenLake uses it to report the probability of the transition that actually happened.

### 2.2 A random-agent baseline

Before training anything we should know what "no learning" looks like. The next cell defines two helpers we reuse for every method in this notebook, `run_episode` (play one episode with a given action function) and `evaluate_policy` (average return over several episodes), together with a `moving_average` used to smooth every learning curve, and then measures the return distribution of a uniformly random policy.

In [ ]:
def moving_average(x, w=20):
    """Trailing moving average; the window grows from 1 to w over the first w points."""
    x = np.asarray(x, dtype=np.float64)
    if len(x) == 0:
        return x
    c = np.cumsum(np.insert(x, 0, 0.0))
    idx = np.arange(len(x))
    lo = np.maximum(0, idx - w + 1)
    return (c[idx + 1] - c[lo]) / (idx + 1 - lo)

def run_episode(env, act_fn, seed=None):
    """Play one episode with act_fn(obs) -> action. Returns (undiscounted return, number of steps)."""
    obs, _ = env.reset(seed=seed)
    total, steps = 0.0, 0
    while True:
        obs, reward, terminated, truncated, _ = env.step(act_fn(obs))
        total += reward
        steps += 1
        if terminated or truncated:
            return total, steps

def evaluate_policy(env, act_fn, n_episodes=20, seed=SEED):
    """Mean and standard deviation of the return of act_fn over n_episodes (the first episode is seeded)."""
    returns = [run_episode(env, act_fn, seed=seed if i == 0 else None)[0] for i in range(n_episodes)]
    return float(np.mean(returns)), float(np.std(returns))

env = make_env("CartPole-v1")
env.action_space.seed(SEED)
random_act = lambda obs: env.action_space.sample()
random_returns = [run_episode(env, random_act, seed=SEED if i == 0 else None)[0] for i in range(300)]
print(f"random policy on CartPole: mean return {np.mean(random_returns):.1f} +- {np.std(random_returns):.1f}, "
      f"min {np.min(random_returns):.0f}, max {np.max(random_returns):.0f}")

plt.figure()
plt.hist(random_returns, bins=30, color="gray", edgecolor="black")
plt.xlabel("episode return (= number of steps the pole stayed up)"); plt.ylabel("episodes")
plt.title("Return distribution of a random policy on CartPole (300 episodes)")
plt.tight_layout(); plt.show()

A random policy keeps the pole up for about 22 steps on average, and almost never beyond 60. The distribution is heavily right-skewed: most episodes end quickly, a few get lucky. This skew is one reason RL learning curves are noisy, and it is the baseline every method below must beat. "Solving" CartPole-v1 is conventionally defined as an average return of at least 475 over 100 consecutive episodes.

> **Common pitfall.** Forgetting that `truncated` ends the episode too. A loop that only checks `terminated` will happily keep stepping a CartPole environment past its 500-step limit: Gymnasium does not raise an error, it simply keeps simulating and keeps reporting `truncated=True`, so the episode lengths and returns you record become meaningless. (Stepping past `terminated` is also allowed but pointless: CartPole logs a warning and returns a reward of 0 for every further step.)

## 3. Tabular methods on FrozenLake

FrozenLake is the simplest environment in which all the ideas of Section 1 can be seen at work. The agent starts at the top-left corner `S` of a 4x4 frozen lake, must reach the goal `G` at the bottom-right, and falls into the lake (episode ends with reward 0) if it steps on a hole `H`. Reaching `G` gives reward 1; every other transition gives 0. With `is_slippery=True` (the default) the ice is slippery: the agent moves in the intended direction only with probability 1/3, and slides to one of the two perpendicular directions with probability 1/3 each. This stochasticity is what makes the problem interesting, because the safest path is not the shortest one.

With 16 states and 4 actions we can store $Q(s, a)$ as a $16 \times 4$ table, hence **tabular** methods. First we inspect the transition model that the environment exposes.

In [ ]:
lake = make_env("FrozenLake-v1", is_slippery=True)
desc = lake.unwrapped.desc                    # 4x4 array of bytes: b'S', b'F', b'H', b'G'
P_lake = lake.unwrapped.P                     # P[s][a] = list of (prob, next_state, reward, terminated)
nS, nA = int(lake.observation_space.n), int(lake.action_space.n)

print("map:")
print("\n".join("".join(ch.decode() for ch in row) for row in desc))
print(f"\n{nS} states, {nA} actions (0 = left, 1 = down, 2 = right, 3 = up)")
print("transitions from state 0 (top-left corner) for action 2 (right):")
for prob, s2, r, done in P_lake[0][2]:
    print(f"   with probability {prob:.3f} -> state {s2:2d}, reward {r:.0f}, terminal {done}")
print("transitions from state 14 (left of the goal) for action 2 (right):")
for prob, s2, r, done in P_lake[14][2]:
    print(f"   with probability {prob:.3f} -> state {s2:2d}, reward {r:.0f}, terminal {done}")

From the corner, "right" leads right (state 1), down (state 4), or, sliding up into the wall, stays in place. From state 14, "right" reaches the goal only a third of the time; the two alternatives, sliding down into the wall (staying in 14) or up to state 10, are both safe here, but from other tiles a slip can land in a hole. The model $P$ is exactly the object in the Bellman equations, so we can run value iteration on it.

### 3.1 Value iteration with the known model

Value iteration applies the optimality operator until the value vector stops changing:

$$
V_{k+1}(s) \;\leftarrow\; \max_a \sum_{s'} P(s' \mid s, a)\, \big[ R(s, a, s') + \gamma\, V_k(s') \big] .
$$

We keep the intermediate $Q_k(s, a)$ (the bracketed expectation for each action) because the policy is read from it. A terminal next state contributes no future value, which the code expresses with the `done` flag; this is the same "no bootstrapping past termination" rule we will use for every method. Convergence is monitored through $\|V_{k+1} - V_k\|_\infty$: once it is below a tolerance, the contraction property guarantees the true error is at most $\gamma/(1-\gamma)$ times that. We also write the plotting helper that draws the value function as a heat-map with the greedy action as an arrow in every cell; it is reused for the learned policies below.

In [ ]:
def value_iteration(P, nS, nA, gamma=0.99, tol=1e-8, max_iter=10_000):
    """Value iteration on a tabular model P[s][a] -> list of (prob, s2, reward, done)."""
    V, deltas = np.zeros(nS), []
    for it in range(max_iter):
        Q = np.zeros((nS, nA))
        for s in range(nS):
            for a in range(nA):
                Q[s, a] = sum(p * (r + (0.0 if done else gamma * V[s2])) for p, s2, r, done in P[s][a])
        V_new = Q.max(axis=1)
        deltas.append(np.abs(V_new - V).max())
        V = V_new
        if deltas[-1] < tol:
            break
    return V, Q, Q.argmax(axis=1), np.array(deltas)

ARROWS = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}     # (dx, dy) in image coordinates: row grows downwards

def plot_policy(V, policy, desc, title, ax=None):
    """Heat-map of V with the greedy action drawn as an arrow; holes are black, the goal is gold."""
    nrow, ncol = desc.shape
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(V.reshape(nrow, ncol), cmap="Blues", vmin=0.0, vmax=max(float(V.max()), 1e-6))
    for r in range(nrow):
        for c in range(ncol):
            s, letter = r * ncol + c, desc[r, c].decode()
            if letter == "H":
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color="black"))
            elif letter == "G":
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color="gold"))
            else:
                dx, dy = ARROWS[int(policy[s])]
                ax.arrow(c - 0.22 * dx, r - 0.22 * dy, 0.4 * dx, 0.4 * dy, head_width=0.16,
                         color="crimson", length_includes_head=True)
                ax.text(c - 0.45, r + 0.42, f"{V[s]:.2f}", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.set_title(title, fontsize=10)

t0 = time.time()
V_star, Q_star, pi_star, vi_deltas = value_iteration(P_lake, nS, nA, gamma=0.99)
print(f"value iteration converged in {len(vi_deltas)} sweeps ({time.time() - t0:.2f}s)")
print("V*(start) =", round(float(V_star[0]), 4))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].semilogy(vi_deltas); axes[0].set_xlabel("sweep"); axes[0].set_ylabel("||V_{k+1} - V_k||_inf")
axes[0].set_title("value iteration convergence")
plot_policy(V_star, pi_star, desc, "optimal policy and V* (gamma = 0.99)", axes[1])
plt.tight_layout(); plt.show()

The optimal policy is not what a person would draw first. In the start state it points *left*, into the wall. On slippery ice, "left" from the corner slides down to state 4 with probability 1/3 and otherwise stays put (the intended move and the "up" slip both hit a wall), so the agent can never be pushed right to state 1, from which a slip downwards lands in the hole at state 5. The same trick appears along the left column and the top row: the policy walks along walls, trading speed for safety. Its value at the start is $V^*(S) \approx 0.54$ for $\gamma = 0.99$. That is not the probability of reaching the goal: every step costs a factor 0.99 and safe routes are long, so the discounted value is well below the raw success probability, which the greedy evaluation at the end of this section measures directly (about 0.74 under the 100-step time limit). With $\gamma = 1$ the undiscounted value would be 0.82, and it is still below 1 because some slips into holes are unavoidable.

> **Try it.** Run `value_iteration` with `gamma=0.9`. The policy changes in the top row (impatience makes shorter, riskier routes look better) and $V^*(S)$ drops to about 0.07, even though the goal is only six steps away.

### 3.2 Q-learning: solving the optimality equation from samples

Value iteration needs $P$. In almost every real problem we do not have it; we can only *sample* transitions by acting. **Temporal-difference** (TD) learning replaces the expectation over $s'$ in the Bellman equation by the single next state we actually observed, and the assignment by a small step towards the sample. For the optimality equation this gives **Q-learning** (Watkins, 1989):

$$
Q(S_t, A_t) \;\leftarrow\; Q(S_t, A_t) + \alpha \Big[ \underbrace{R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a')}_{\text{TD target}} - Q(S_t, A_t) \Big].
$$

The quantity in brackets is the **TD error** $\delta_t$: the difference between a one-step bootstrapped estimate of the return (the target) and the current estimate. If the target is on average equal to the right-hand side of the Bellman optimality equation, then the update is a stochastic approximation of value iteration, and indeed Q-learning converges to $Q^*$ provided every state–action pair is visited infinitely often and the learning rate $\alpha$ decays appropriately. A remarkable feature is that convergence holds *no matter which policy generates the data*, as long as it explores enough: Q-learning is **off-policy**. The max in the target evaluates the greedy policy, whatever the behaviour policy (here $\varepsilon$-greedy) is doing.

### 3.3 SARSA: the on-policy alternative

If we replace the max by the action the agent *actually takes next*, $A_{t+1} \sim \pi(\cdot \mid S_{t+1})$, we obtain **SARSA** (named after the tuple $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$ it consumes):

$$
Q(S_t, A_t) \;\leftarrow\; Q(S_t, A_t) + \alpha \big[ R_{t+1} + \gamma\, Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \big].
$$

This is a sample of the Bellman *expectation* equation for the behaviour policy, so SARSA learns $Q^\pi$ of the $\varepsilon$-greedy policy it follows: it is **on-policy**. The practical difference is that SARSA "knows" it will keep exploring and therefore learns a policy that is safe *given* that it will occasionally take random actions, while Q-learning learns the policy that is best *if* it stopped exploring. In the classic cliff-walking example SARSA takes a longer route far from the cliff and Q-learning walks along the edge. As $\varepsilon \to 0$ the two coincide. The implementation below handles both methods with one function whose only difference is one line of the target.

In [ ]:
def epsilon_greedy(Q, s, eps, rng):
    """Random action with probability eps, otherwise a greedy action (ties broken at random)."""
    if rng.random() < eps:
        return int(rng.integers(Q.shape[1]))
    q = Q[s]
    return int(rng.choice(np.flatnonzero(q == q.max())))

def linear_eps(n_episodes, eps_start=1.0, eps_end=0.05, fraction=0.5):
    """Epsilon schedule: linear decay from eps_start to eps_end over the first `fraction` of the episodes."""
    decay_len = max(1, int(fraction * n_episodes))
    return lambda ep: max(eps_end, eps_start - (eps_start - eps_end) * ep / decay_len)

def tabular_td(env, n_episodes, method="q_learning", alpha=0.1, gamma=0.99, eps_fn=None, seed=SEED):
    """Tabular Q-learning (off-policy) or SARSA (on-policy) with epsilon-greedy exploration.

    Returns the Q-table and an array with the return of every training episode (1 = reached the goal).
    """
    assert method in ("q_learning", "sarsa")
    rng = np.random.default_rng(seed)
    nS, nA = int(env.observation_space.n), int(env.action_space.n)
    Q = np.zeros((nS, nA))
    eps_fn = eps_fn or linear_eps(n_episodes)
    ep_returns = np.zeros(n_episodes)
    for ep in range(n_episodes):
        eps = eps_fn(ep)
        s, _ = env.reset(seed=seed if ep == 0 else None)
        a = epsilon_greedy(Q, s, eps, rng)
        while True:
            s2, r, terminated, truncated, _ = env.step(a)
            a2 = epsilon_greedy(Q, s2, eps, rng)          # the action the agent will actually take next
            if terminated:
                target = r                                # no future beyond a terminal state
            elif method == "q_learning":
                target = r + gamma * Q[s2].max()          # evaluates the greedy policy (off-policy)
            else:
                target = r + gamma * Q[s2, a2]            # evaluates the behaviour policy (on-policy)
            Q[s, a] += alpha * (target - Q[s, a])
            ep_returns[ep] += r
            s, a = s2, a2
            if terminated or truncated:
                break
    return Q, ep_returns

N_EPISODES_TAB = 3_000 if FAST_RUN else 10_000
tab_results = {}
for method in ["q_learning", "sarsa"]:
    t0 = time.time()
    Q_m, rets = tabular_td(lake, N_EPISODES_TAB, method=method)
    tab_results[method] = (Q_m, rets)
    print(f"{method:10s}: success rate over the last 500 training episodes = {rets[-500:].mean():.3f}"
          f"   ({time.time() - t0:.1f}s)")

plt.figure()
for method, (Q_m, rets) in tab_results.items():
    plt.plot(moving_average(rets, 200), label=method)
plt.axhline(evaluate_policy(lake, lambda s: int(pi_star[s]), n_episodes=500)[0], color="k", ls="--",
            label="optimal policy (value iteration)")
plt.xlabel("training episode"); plt.ylabel("success rate (moving average, 200 episodes)")
plt.title("Q-learning and SARSA on slippery FrozenLake"); plt.legend(); plt.tight_layout(); plt.show()

Both curves start at the random-policy success rate (one to two percent), climb as $\varepsilon$ decays, and settle below the optimal policy's success rate because the training episodes still include 5% random actions. The curves are noisy even after smoothing over 200 episodes: on slippery ice a single episode succeeds or fails largely by chance, so the *policy* can be near-optimal while the per-episode outcome remains a coin flip with a loaded coin.

To compare what was actually learned we look at the greedy policies, side by side with the value-iteration solution, and evaluate each greedy policy with exploration switched off.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
plot_policy(V_star, pi_star, desc, "value iteration (known model)", axes[0])
for ax, method in zip(axes[1:], ["q_learning", "sarsa"]):
    Q_m = tab_results[method][0]
    plot_policy(Q_m.max(axis=1), Q_m.argmax(axis=1), desc, f"{method} (learned from samples)", ax)
plt.tight_layout(); plt.show()

N_EVAL_TAB = 1_000
print(f"greedy success rate over {N_EVAL_TAB} evaluation episodes:")
print(f"   value iteration : {evaluate_policy(lake, lambda s: int(pi_star[s]), N_EVAL_TAB)[0]:.3f}")
for method in ["q_learning", "sarsa"]:
    pol = tab_results[method][0].argmax(axis=1)
    print(f"   {method:16s}: {evaluate_policy(lake, lambda s: int(pol[s]), N_EVAL_TAB)[0]:.3f}")

Q-learning recovers the value-iteration policy almost everywhere (differences appear only in states whose actions are nearly tied, or that are rarely visited), and its greedy success rate matches the optimal one within sampling noise. The learned values are typically *lower* than $V^*$ because the table is still being updated from an $\varepsilon$-greedy stream and because the linear schedule keeps $\varepsilon$ at 0.05, but the *ordering* of actions, which is all that matters for the greedy policy, is right. SARSA's policy tends to differ in the cells next to holes, where it prefers actions that remain safe even if the next action is random; its greedy success rate is usually a little lower on this map, which is the price of learning the value of a cautious policy rather than the optimal one. Exercise 1 repeats this comparison on the non-slippery lake, where the two methods agree exactly.

> **Try it.** Set `alpha=0.5` in `tabular_td`. The curves rise faster and become noisier; on a stochastic environment a large learning rate never lets the estimates average out over the random transitions.

Tabular methods are exact and provably convergent, but they need one entry per state. CartPole's state is a vector of four real numbers; discretizing it finely enough for control would need millions of cells, and any interesting problem (images, robots, text) has an astronomically large or continuous state space in which the agent never visits the same state twice. What we need is *generalization* across states, which is what neural networks are for. The rest of this notebook replaces the table by a network, first for the policy (Section 4) and then for the Q-function (Section 5).

Notebook 13 · Reinforcement Learning — part 2 (policy gradients / REINFORCE, DQN / Double DQN)

## 4. Policy gradients: REINFORCE

### 4.1 Learning the policy directly

The tabular methods learned a value function and derived the policy from it by taking an arg-max. **Policy gradient** methods take the opposite route: they parametrize the policy directly, $\pi_\theta(a \mid s)$, as a neural network with parameters $\theta$ that outputs the logits of a categorical distribution over actions, and they adjust $\theta$ by gradient *ascent* on the expected return. The advantages are that the policy can be stochastic (useful for exploration and in partially observed problems), that it extends naturally to continuous actions (output the mean and variance of a Gaussian), and that a small change in $\theta$ produces a small change in behaviour, unlike the arg-max of a Q-function, which can flip abruptly.

Let $\tau = (s_0, a_0, r_1, s_1, a_1, \dots, s_{T})$ denote a trajectory and $R(\tau) = \sum_{t=0}^{T-1} \gamma^t r_{t+1}$ its discounted return. Under policy $\pi_\theta$ the trajectory has probability

$$
p_\theta(\tau) = p(s_0) \prod_{t=0}^{T-1} \pi_\theta(a_t \mid s_t)\, P(s_{t+1} \mid s_t, a_t),
$$

and the objective is $J(\theta) = \mathbb{E}_{\tau \sim p_\theta}[R(\tau)]$. The difficulty is that $\theta$ enters through the *sampling distribution*, not through a differentiable function of the sample, and the transition kernel $P$ is an unknown black box. We cannot write `J.backward()`.

### 4.2 The policy gradient theorem

The log-derivative trick resolves both difficulties. For any density $p_\theta$ and any function $f$ that does not depend on $\theta$,

$$
\nabla_\theta \mathbb{E}_{\tau \sim p_\theta}[f(\tau)]
= \int \nabla_\theta p_\theta(\tau)\, f(\tau)\, d\tau
= \int p_\theta(\tau)\, \nabla_\theta \log p_\theta(\tau)\, f(\tau)\, d\tau
= \mathbb{E}_{\tau \sim p_\theta}\big[ \nabla_\theta \log p_\theta(\tau)\, f(\tau) \big],
$$

using $\nabla p = p \nabla \log p$. The expectation on the right can be estimated by sampling trajectories, which is all we can do with a black-box environment. Now take the log of $p_\theta(\tau)$: it is a *sum* of log-probabilities, and the terms that involve $P$ and $p(s_0)$ do not depend on $\theta$, so their gradient vanishes,

$$
\nabla_\theta \log p_\theta(\tau) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t).
$$

The unknown dynamics have dropped out entirely. Substituting gives the **policy gradient theorem** in its trajectory form (Williams, 1992; Sutton et al., 2000):

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim p_\theta}\Big[ \Big( \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t) \Big) R(\tau) \Big].
$$

The estimator has a clean interpretation: it increases the log-probability of every action taken in a trajectory in proportion to how good the whole trajectory turned out to be. Actions in good trajectories become more likely, actions in bad ones less likely. The algorithm that samples one or a few trajectories, computes this quantity and takes a gradient-ascent step is **REINFORCE**.

### 4.3 Reward-to-go and baselines: reducing variance without adding bias

REINFORCE in its raw form is unbiased but extraordinarily noisy, and two refinements are essential in practice. Both leave the expectation unchanged and shrink the variance.

**Reward-to-go.** The action at time $t$ cannot influence rewards received *before* $t$, and indeed one can show that $\mathbb{E}[\nabla_\theta \log \pi_\theta(a_t \mid s_t)\, r_{t'}] = 0$ for $t' \le t$ (condition on $s_t$ and note that $\mathbb{E}_{a_t \sim \pi}[\nabla \log \pi(a_t \mid s_t)] = \nabla \sum_a \pi(a \mid s_t) = \nabla 1 = 0$, while $r_{t'}$ for $t' \le t$ is already determined). Multiplying each log-gradient by the full return therefore adds terms with zero mean and positive variance. Dropping them gives the **reward-to-go** estimator, which weights each action by the return *from that step onwards*:

$$
\nabla_\theta J(\theta) = \mathbb{E}\Big[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t)\, G_t \Big], \qquad G_t = \sum_{k=0}^{T-1-t} \gamma^k r_{t+k+1}.
$$

(Strictly speaking, for the discounted objective $J$ the term for step $t$ also carries a factor $\gamma^t$, because a reward at step $t+k+1$ enters $R(\tau)$ with weight $\gamma^{t+k}$. Virtually every implementation drops this factor, which amounts to treating $\gamma$ inside $G_t$ as a variance-reduction device rather than as part of the objective; we follow that convention here and in the actor-critic code of Section 6.)

**Baselines.** The same identity $\mathbb{E}_{a \sim \pi(\cdot \mid s)}[\nabla_\theta \log \pi_\theta(a \mid s)] = 0$ shows that we may subtract from $G_t$ any function $b(s_t)$ of the state, as long as it does not depend on the action:

$$
\mathbb{E}_{a_t}\big[ \nabla_\theta \log \pi_\theta(a_t \mid s_t)\, b(s_t) \big] = b(s_t)\, \nabla_\theta \sum_a \pi_\theta(a \mid s_t) = 0 .
$$

The estimator $\sum_t \nabla \log \pi(a_t \mid s_t)\,(G_t - b(s_t))$ is still unbiased, but its variance can be much smaller. The intuition is that what matters for learning is whether an action was *better or worse than typical* in its state, not whether the absolute return was large. In CartPole every return is positive, so without a baseline REINFORCE increases the probability of *every* action it took, including the bad ones, and relies on good actions being increased slightly more; with a baseline, actions that led to below-average returns are actively discouraged. The natural choice is $b(s) = V^\pi(s)$, the state-value function, which makes $G_t - b(s_t)$ an estimate of the **advantage** $A^\pi(s_t, a_t) = Q^\pi(s_t, a_t) - V^\pi(s_t)$. We learn it with a second small network trained by regression on the observed returns, exactly as in supervised learning.

> **Key idea.** The policy gradient is an expectation we can sample; reward-to-go and a state-dependent baseline are the two standard ways to lower the variance of the sample without changing its mean.

The implementation below turns the gradient estimate into a *surrogate loss* that autograd can differentiate: $L(\theta) = -\frac{1}{T} \sum_t \log \pi_\theta(a_t \mid s_t)\, w_t$, where the weights $w_t$ (returns or advantages) are treated as constants. Its gradient is exactly the negative of the estimator, so a step of Adam on $L$ is a step of gradient ascent on $J$. Note the `.detach()` on the value network's prediction: the baseline must be a constant in the policy loss. The policy gradient itself would not change without it (the value network's parameters do not enter $\pi_\theta$), but the policy loss would then also send a spurious gradient into the value network, pushing $v$ in whatever direction makes $\log \pi_\theta(a_t \mid s_t)\,(G_t - v)$ smaller instead of making $v$ track the returns, and the baseline would drift away from $V^\pi$. The value network gets its own loss, a mean-squared error to the returns.

In [ ]:
def to_tensor(x):
    """NumPy observation (or a Python list) -> float32 tensor on the training device."""
    return torch.as_tensor(np.asarray(x, dtype=np.float32), device=device)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

class PolicyNet(nn.Module):
    """MLP mapping an observation to action logits."""
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, n_actions))
    def forward(self, x):
        return self.net(x)

class ValueNet(nn.Module):
    """MLP mapping an observation to a scalar state-value estimate."""
    def __init__(self, obs_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def collect_episode(env, policy, seed=None):
    """Sample one episode from a stochastic policy. Returns (observations, actions, rewards)."""
    obs, _ = env.reset(seed=seed)
    obs_buf, act_buf, rew_buf = [], [], []
    while True:
        obs_t = to_tensor(obs)
        with torch.no_grad():
            action = int(Categorical(logits=policy(obs_t)).sample().item())
        obs, reward, terminated, truncated, _ = env.step(action)
        obs_buf.append(obs_t); act_buf.append(action); rew_buf.append(float(reward))
        if terminated or truncated:
            return torch.stack(obs_buf), torch.as_tensor(act_buf, device=device), np.array(rew_buf)

def reinforce(env, n_episodes, gamma=0.99, lr=5e-3, reward_to_go=True, baseline=True, seed=SEED, log_every=50):
    """REINFORCE with optional reward-to-go weighting and a learned state-value baseline.

    Returns (per-episode returns, cumulative environment steps after each episode, the trained policy).
    """
    seed_everything(seed)
    obs_dim, n_actions = env.observation_space.shape[0], int(env.action_space.n)
    policy = PolicyNet(obs_dim, n_actions).to(device)
    value = ValueNet(obs_dim).to(device) if baseline else None
    params = list(policy.parameters()) + (list(value.parameters()) if baseline else [])
    opt = torch.optim.Adam(params, lr=lr)
    ep_returns, steps_at, total_steps, t0 = [], [], 0, time.time()
    for ep in range(n_episodes):
        S, A, rewards = collect_episode(env, policy, seed=seed if ep == 0 else None)
        G = discounted_returns(rewards, gamma)
        if not reward_to_go:
            G = np.full(len(G), G[0])                    # every action is weighted by the whole-episode return
        G_t = torch.as_tensor(G, dtype=torch.float32, device=device)
        log_prob = Categorical(logits=policy(S)).log_prob(A)
        if baseline:
            v = value(S)
            weights = G_t - v.detach()                   # advantage estimate; the baseline is a constant here
            value_loss = F.mse_loss(v, G_t)
        else:
            weights = G_t
            value_loss = torch.zeros((), device=device)
        policy_loss = -(log_prob * weights).mean()
        loss = policy_loss + 0.5 * value_loss
        opt.zero_grad(); loss.backward(); opt.step()
        total_steps += len(rewards)
        ep_returns.append(float(rewards.sum())); steps_at.append(total_steps)
        if (ep + 1) % log_every == 0:
            print(f"episode {ep + 1:4d} | env steps {total_steps:7d} | mean return (last {log_every}) "
                  f"{np.mean(ep_returns[-log_every:]):6.1f} | {time.time() - t0:5.1f}s")
    return np.array(ep_returns), np.array(steps_at), policy

We now train two agents with identical seeds and budgets: the raw estimator (whole-episode return, no baseline) and the refined one (reward-to-go with a learned baseline). Under `FAST_RUN` the budget is only 150 episodes per variant, enough to see the difference in learning speed but not to solve the task; the full run uses 600 episodes, which is usually enough for the refined variant to reach returns near 500. (A note on time: every method in this notebook runs small networks on a single environment, so the cost is dominated by Python-side environment steps rather than by matrix multiplications, and a GPU does not speed things up much; the full run takes on the order of 20 minutes on a CPU.)

In [ ]:
N_EPISODES_PG = 150 if FAST_RUN else 600
env = make_env("CartPole-v1")
pg_results = {}
for name, kwargs in [("full return, no baseline", dict(reward_to_go=False, baseline=False)),
                     ("reward-to-go + baseline", dict(reward_to_go=True, baseline=True))]:
    print(f"--- REINFORCE: {name}")
    rets, steps, pol = reinforce(env, N_EPISODES_PG, log_every=50 if FAST_RUN else 100, **kwargs)
    pg_results[name] = (rets, steps, pol)

plt.figure()
for name, (rets, steps, _) in pg_results.items():
    plt.plot(rets, alpha=0.25)
    plt.plot(moving_average(rets, 20), label=name)
plt.xlabel("episode"); plt.ylabel("return (moving average of 20)")
plt.title("REINFORCE on CartPole: variance reduction speeds up learning"); plt.legend(); plt.tight_layout(); plt.show()

Both agents improve over the random baseline, but the refined estimator climbs faster and more steadily; the raw estimator often stalls for long stretches or collapses after a run of lucky episodes pushed all its action probabilities in one direction. The faint lines show the per-episode returns behind the moving averages: even the better agent's return jumps by hundreds from one episode to the next, which is why we never judge an RL run by a single episode.

Note that the x-axis is *episodes*, while the cost of training is *environment steps*, and better agents have longer episodes. When we compare all methods in Section 6.5 we will plot returns against steps.

### 4.4 Measuring the variance of the gradient estimator

The claim that reward-to-go and baselines reduce variance can be checked directly. We freeze the policy trained above, sample 40 episodes, and compute the single-episode gradient estimate under three estimators: full return, reward-to-go, and reward-to-go with a constant baseline (the average reward-to-go, estimated on a separate set of episodes so that it does not depend on the actions being scored). Because the seed is reset before each variant, all three estimators are evaluated on exactly the same 40 trajectories, so the comparison is paired. The quantity of interest is the total variance $\sum_i \mathrm{Var}[\hat g_i]$ of the flattened gradient, relative to the norm of its mean, a signal-to-noise ratio for one episode of data.

In [ ]:
def policy_gradient_samples(policy, env, n_episodes, gamma, variant, baseline_value=0.0, seed=SEED):
    """One REINFORCE gradient estimate per episode (flattened over all parameters), for a given estimator."""
    seed_everything(seed)
    grads = []
    for ep in range(n_episodes):
        S, A, rewards = collect_episode(env, policy, seed=seed if ep == 0 else None)
        G = discounted_returns(rewards, gamma)
        if variant == "full return":
            w = np.full(len(G), G[0])
        elif variant == "reward-to-go":
            w = G
        else:
            w = G - baseline_value
        W = torch.as_tensor(w, dtype=torch.float32, device=device)
        policy.zero_grad()
        (-(Categorical(logits=policy(S)).log_prob(A) * W).sum()).backward()
        grads.append(torch.cat([p.grad.flatten() for p in policy.parameters()]).clone())
    return torch.stack(grads)

probe_policy = pg_results["reward-to-go + baseline"][2]
seed_everything(SEED + 1)
calib_G = np.concatenate([discounted_returns(collect_episode(env, probe_policy)[2], 0.99) for _ in range(20)])
b_const = float(calib_G.mean())
print(f"constant baseline (mean reward-to-go on 20 calibration episodes): {b_const:.1f}\n")

N_GRAD_EPISODES = 40
print(f"{'estimator':34s} {'||mean grad||':>14s} {'total variance':>15s} {'noise / signal':>15s}")
for variant, b in [("full return", 0.0), ("reward-to-go", 0.0), ("reward-to-go + constant baseline", b_const)]:
    grads = policy_gradient_samples(probe_policy, env, N_GRAD_EPISODES, 0.99, variant, baseline_value=b)
    mean_norm = grads.mean(0).norm().item()
    total_var = grads.var(0).sum().item()
    print(f"{variant:34s} {mean_norm:14.2f} {total_var:15.1f} {math.sqrt(total_var) / mean_norm:15.2f}")

The three estimators agree on the *direction* of the gradient (the norms of the mean gradient are similar, as they must be for unbiased estimators of the same quantity), but their noise differs by a large factor: reward-to-go removes the terms that pair an action with rewards it could not have influenced, and the baseline removes the common positive offset of all returns. The last column is the ratio of the standard deviation of a one-episode estimate to the norm of the true gradient; a value of 5 means a single episode tells us almost nothing about the direction to move and we would need on the order of 25 episodes to get a step with more signal than noise. This is the fundamental weakness of Monte Carlo policy gradients: they are unbiased but need a lot of data. The two methods that follow trade some bias for a lot less variance, first by bootstrapping values (DQN) and then by combining both ideas (actor-critic).

> **Try it.** Replace `probe_policy` with a freshly initialized `PolicyNet(4, 2).to(device)`. Episodes are then short and all estimators look similar; the benefit of variance reduction grows with the episode length because the number of "irrelevant" reward terms grows quadratically with $T$.

## 5. Deep Q-Networks

### 5.1 From the table to a network, and the deadly triad

Q-learning needs a table with one entry per state–action pair. To scale it to CartPole we replace the table by a network $Q_\theta(s, \cdot)$ that takes the observation and outputs one value per action, and we replace the tabular update by a gradient step on the squared TD error,

$$
L(\theta) = \Big( \underbrace{r + \gamma \max_{a'} Q_\theta(s', a')}_{\text{target } y} - Q_\theta(s, a) \Big)^2 ,
$$

treating the target as a constant, as the tabular update does. This is a *regression* problem, except that the regression targets are produced by the network being trained and change every step. Sutton and Barto call the combination of (1) function approximation, (2) bootstrapping (targets built from the network's own predictions) and (3) off-policy learning (data from an $\varepsilon$-greedy or replayed policy, target evaluating the greedy one) the **deadly triad**: with all three present the contraction argument of Section 1 no longer applies and the values can diverge. Nothing guarantees that a gradient step that fixes $Q(s, a)$ does not simultaneously ruin $Q(s', a')$ and hence the target that was just used.

The **Deep Q-Network** (DQN) of Mnih et al. (2015) made this combination work in practice with two mechanisms, plus a few engineering details, and the next subsection introduces them one by one.

### 5.2 Experience replay, target networks, exploration schedules and the Huber loss

**Experience replay.** Consecutive transitions of a trajectory are strongly correlated, and the gradient of a mini-batch of consecutive transitions is a poor estimate of the gradient of the loss over the whole state distribution. DQN stores every transition $(s, a, r, s', d)$ in a large ring buffer and trains on uniformly sampled mini-batches from it. This breaks the temporal correlation, brings the data closer to the i.i.d. setting that SGD assumes, and lets each transition be reused many times. Replay is only possible *because* Q-learning is off-policy: the transitions were generated by old versions of the policy, but the target still evaluates the current greedy policy.

**Target network.** To stop the target from moving with every gradient step, DQN keeps a frozen copy $\theta^-$ of the parameters that is updated only every $C$ steps (here $C = 500$), and computes the target with it: $y = r + \gamma (1 - d) \max_{a'} Q_{\theta^-}(s', a')$. For $C$ steps the regression problem is stationary and the network can actually fit it. The factor $(1 - d)$ implements "no bootstrapping past termination" ($d = 1$ if $s'$ is terminal). Here the distinction between `terminated` and `truncated` matters: a transition that was cut by the time limit is stored with $d = 0$, because the state after it is a perfectly ordinary state from which the return would have continued. Storing it with $d = 1$ would teach the network that states reached late in an episode are worth nothing, which on CartPole visibly hurts.

**Exploration schedule.** Actions are $\varepsilon$-greedy with respect to the online network, with $\varepsilon$ decaying linearly from 1.0 to 0.05 over the first part of training. Training starts only once the buffer holds a few hundred transitions.

**Huber loss.** Early in training the TD errors can be large, and a squared error then produces huge gradients that destabilize the network. DQN uses the Huber loss, which is quadratic for small errors and linear beyond a threshold $\delta$,

$$
\ell_\delta(u) = \begin{cases} \tfrac{1}{2} u^2 & |u| \le \delta \\ \delta\,(|u| - \tfrac{1}{2}\delta) & \text{otherwise,} \end{cases}
$$

so that the gradient magnitude is capped at $\delta$. In PyTorch, `F.smooth_l1_loss` with its default `beta=1.0` is exactly $\ell_1$. We add gradient-norm clipping as a second safety net.

The implementation below keeps the buffer in pre-allocated NumPy arrays (faster and more memory-efficient than a `deque` of tuples) and exposes an `update_priorities` hook that is a no-op here and is used by the prioritized replay of Exercise 3. The `double` flag switches to the Double DQN target of Section 5.3; the `net_cls` argument lets Exercise 5 plug in a dueling architecture.

In [ ]:
class ReplayBuffer:
    """Uniform experience replay in pre-allocated NumPy arrays (a ring buffer)."""
    def __init__(self, capacity, obs_dim):
        self.obs = np.zeros((capacity, obs_dim), np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), np.float32)
        self.actions = np.zeros(capacity, np.int64)
        self.rewards = np.zeros(capacity, np.float32)
        self.dones = np.zeros(capacity, np.float32)
        self.capacity, self.ptr, self.size = capacity, 0, 0

    def add(self, obs, action, reward, next_obs, done):
        i = self.ptr
        self.obs[i], self.actions[i], self.rewards[i] = obs, action, reward
        self.next_obs[i], self.dones[i] = next_obs, done
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, rng):
        """Uniform mini-batch. Returns ((S, A, R, S2, D), indices, importance weights) - weights are all 1 here."""
        idx = rng.integers(self.size, size=batch_size)
        batch = (torch.as_tensor(self.obs[idx], device=device),
                 torch.as_tensor(self.actions[idx], device=device),
                 torch.as_tensor(self.rewards[idx], device=device),
                 torch.as_tensor(self.next_obs[idx], device=device),
                 torch.as_tensor(self.dones[idx], device=device))
        return batch, idx, torch.ones(batch_size, device=device)

    def update_priorities(self, idx, td_errors):
        """Hook used by prioritized replay (Exercise 3); uniform replay ignores it."""
        pass

    def __len__(self):
        return self.size


class QNetwork(nn.Module):
    """MLP mapping an observation to one Q-value per action."""
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_actions))
    def forward(self, x):
        return self.net(x)


def greedy_act(q_net):
    """Wrap a Q-network into an act(obs) -> action function (no exploration)."""
    def act(obs):
        with torch.no_grad():
            return int(q_net(to_tensor(obs)).argmax().item())
    return act


def dqn(env, total_steps, double=False, gamma=0.99, lr=1e-3, batch_size=64, buffer_size=50_000,
        learning_starts=500, target_update=500, eps_start=1.0, eps_end=0.05, eps_decay_steps=10_000,
        net_cls=QNetwork, buffer_cls=ReplayBuffer, solve_threshold=475.0, seed=SEED, log_every=2_000):
    """(Double) DQN with experience replay, a target network, the Huber loss and gradient clipping.

    Returns (online Q-network, per-episode returns, env steps at the end of each episode, training losses).
    Training stops early once the mean return of the last 20 training episodes reaches solve_threshold.
    """
    seed_everything(seed)
    rng = np.random.default_rng(seed)
    obs_dim, n_actions = env.observation_space.shape[0], int(env.action_space.n)
    q_net = net_cls(obs_dim, n_actions).to(device)
    q_target = copy.deepcopy(q_net)
    for p in q_target.parameters():
        p.requires_grad_(False)
    opt = torch.optim.Adam(q_net.parameters(), lr=lr)
    buffer = buffer_cls(buffer_size, obs_dim)
    ep_returns, steps_at, losses = [], [], []
    obs, _ = env.reset(seed=seed)
    ep_return, t0 = 0.0, time.time()
    for step in range(1, total_steps + 1):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * step / eps_decay_steps)
        if rng.random() < eps:
            action = int(rng.integers(n_actions))
        else:
            with torch.no_grad():
                action = int(q_net(to_tensor(obs)).argmax().item())
        next_obs, reward, terminated, truncated, _ = env.step(action)
        buffer.add(obs, action, reward, next_obs, float(terminated))   # truncation is not terminal: keep bootstrapping
        ep_return += reward
        obs = next_obs
        if terminated or truncated:
            ep_returns.append(ep_return); steps_at.append(step); ep_return = 0.0
            obs, _ = env.reset()
            if len(ep_returns) >= 20 and np.mean(ep_returns[-20:]) >= solve_threshold:
                print(f"solved: mean return of the last 20 training episodes >= {solve_threshold:.0f} after {step} steps")
                break
        if step >= learning_starts:
            (S, A, R, S2, D), idx, w = buffer.sample(batch_size, rng)
            q_sa = q_net(S).gather(1, A.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                if double:
                    best_next = q_net(S2).argmax(dim=1, keepdim=True)        # SELECT with the online network
                    q_next = q_target(S2).gather(1, best_next).squeeze(1)     # EVALUATE with the target network
                else:
                    q_next = q_target(S2).max(dim=1).values                    # select and evaluate with the target
                target = R + gamma * (1.0 - D) * q_next
            td_error = target - q_sa
            loss = (w * F.smooth_l1_loss(q_sa, target, reduction="none")).mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
            opt.step()
            buffer.update_priorities(idx, td_error.detach().abs().cpu().numpy())
            losses.append(loss.item())
        if step % target_update == 0:
            q_target.load_state_dict(q_net.state_dict())
        if step % log_every == 0:
            recent = np.mean(ep_returns[-20:]) if ep_returns else float("nan")
            print(f"step {step:6d} | eps {eps:.3f} | episodes {len(ep_returns):4d} | "
                  f"mean return (last 20) {recent:6.1f} | {time.time() - t0:5.1f}s")
    return q_net, np.array(ep_returns), np.array(steps_at), np.array(losses)

A few implementation details deserve attention. `gather(1, A.unsqueeze(1))` picks, from the $(B, 2)$ matrix of Q-values, the value of the action that was actually taken in each transition; the network outputs values for *all* actions but only one of them receives a gradient. The whole target computation sits under `torch.no_grad()`, which both saves memory and enforces that the target is a constant. The loss is written as an elementwise Huber loss multiplied by per-sample weights and then averaged; with uniform replay the weights are all one, so this is the ordinary mean Huber loss, but the same code supports the importance weights of prioritized replay. Finally, the target network is refreshed by copying the `state_dict`, the simplest form of the "hard" update; many later implementations instead use a slowly moving Polyak average $\theta^- \leftarrow (1 - \tau)\theta^- + \tau\theta$ with $\tau \approx 0.005$.

We now train DQN on CartPole. Under `FAST_RUN` we allow 6,000 environment steps with the exploration schedule compressed to 2,000 steps, which is enough to see learning begin; the full run uses 60,000 steps and stops early once the training returns average 475 or more.

In [ ]:
DQN_STEPS = 6_000 if FAST_RUN else 60_000
DQN_EPS_DECAY = 2_000 if FAST_RUN else 10_000
DQN_LOG = 1_000 if FAST_RUN else 5_000
env = make_env("CartPole-v1")

print("--- DQN")
q_dqn, ret_dqn, steps_dqn, loss_dqn = dqn(env, DQN_STEPS, double=False, eps_decay_steps=DQN_EPS_DECAY, log_every=DQN_LOG)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(steps_dqn, ret_dqn, alpha=0.3, label="episode return")
axes[0].plot(steps_dqn, moving_average(ret_dqn, 20), label="moving average (20)")
axes[0].set_xlabel("environment steps"); axes[0].set_ylabel("return"); axes[0].set_title("DQN on CartPole"); axes[0].legend()
axes[1].plot(moving_average(loss_dqn, 100)); axes[1].set_xlabel("gradient step"); axes[1].set_ylabel("Huber loss (moving average)")
axes[1].set_title("TD loss during training")
plt.tight_layout(); plt.show()

The return curve is flat during the exploration phase, then rises, often in bursts, once $\varepsilon$ is small and the target network has caught up with a useful value function. The loss curve deserves a comment because it behaves unlike a supervised loss: it *increases* as the agent improves. Longer episodes mean larger returns, hence larger Q-values and larger absolute TD errors, and every refresh of the target network shifts the regression targets and produces a small spike. A rising TD loss is therefore not a sign of failure; the return is the metric that matters.

### 5.3 Overestimation and Double DQN

The max in the Q-learning target introduces a systematic bias. Suppose the estimates $Q(s', a')$ are noisy, $Q(s', a') = Q^*(s', a') + \epsilon_{a'}$ with zero-mean noise. Then, because the maximum is a convex function, Jensen's inequality gives

$$
\mathbb{E}\Big[ \max_{a'} \big( Q^*(s', a') + \epsilon_{a'} \big) \Big] \;\ge\; \max_{a'} \mathbb{E}\big[ Q^*(s', a') + \epsilon_{a'} \big] = \max_{a'} Q^*(s', a').
$$

The target is biased upwards, and since the network fits the target, the overestimation propagates and compounds through bootstrapping. The problem is that the same noisy estimate is used both to *choose* the best next action and to *evaluate* it: whichever action happens to have the most positive noise is selected, and its inflated value is used. **Double DQN** (van Hasselt et al., 2016) decouples the two roles. The online network chooses the action and the target network evaluates it,

$$
y^{\text{DDQN}} = r + \gamma\, (1 - d)\, Q_{\theta^-}\Big( s', \arg\max_{a'} Q_\theta(s', a') \Big).
$$

Because the two networks have different noise, an action selected for its lucky noise in $Q_\theta$ is not automatically also lucky in $Q_{\theta^-}$, and the bias largely cancels. The change is two lines of code (the `double=True` branch above), it costs nothing, and it is used by essentially every modern value-based agent.

We train the Double DQN variant with the same seed and budget, compare the learning curves, evaluate both greedy policies without exploration, and check for overestimation directly by comparing each network's predicted value of the start state, $\max_a Q(s_0, a)$, with the discounted return the greedy policy actually obtains from $s_0$.

In [ ]:
print("--- Double DQN")
q_ddqn, ret_ddqn, steps_ddqn, loss_ddqn = dqn(env, DQN_STEPS, double=True, eps_decay_steps=DQN_EPS_DECAY, log_every=DQN_LOG)

plt.figure()
plt.plot(steps_dqn, moving_average(ret_dqn, 20), label="DQN")
plt.plot(steps_ddqn, moving_average(ret_ddqn, 20), label="Double DQN")
plt.xlabel("environment steps"); plt.ylabel("return (moving average of 20 episodes)")
plt.title("DQN vs Double DQN on CartPole"); plt.legend(); plt.tight_layout(); plt.show()

def start_value_check(q_net, env, n_episodes=10, gamma=0.99, seed=SEED):
    """Mean of max_a Q(s_0, a) vs the mean realized discounted return of the greedy policy from s_0."""
    act, est, real = greedy_act(q_net), [], []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed if i == 0 else None)
        with torch.no_grad():
            est.append(q_net(to_tensor(obs)).max().item())
        rewards = []
        while True:
            obs, r, terminated, truncated, _ = env.step(act(obs))
            rewards.append(r)
            if terminated or truncated:
                break
        real.append(discounted_returns(np.array(rewards), gamma)[0])
    return float(np.mean(est)), float(np.mean(real))

print(f"{'agent':12s} {'greedy return (10 ep.)':>24s} {'predicted V(s0)':>16s} {'realized disc. return':>22s}")
for name, q_net in [("DQN", q_dqn), ("Double DQN", q_ddqn)]:
    mean_ret, std_ret = evaluate_policy(env, greedy_act(q_net), n_episodes=10)
    est, real = start_value_check(q_net, env)
    print(f"{name:12s} {mean_ret:15.1f} +- {std_ret:5.1f} {est:16.2f} {real:22.2f}")

On a problem as small as CartPole the two variants often reach similar returns, and the more informative numbers are in the last two columns. A discount factor of 0.99 and a reward of 1 per step cap the true value of any state at $1/(1-\gamma) = 100$, so the two columns can be read on an absolute scale. Early in training, and therefore throughout the short `FAST_RUN` budget, both networks *under*-estimate: values propagate backwards from the end of episodes one bootstrap step at a time, the target network delays the propagation further, and the predictions lag far behind the returns the greedy policy already achieves. Overestimation is a late-training phenomenon that appears once the values have caught up and the noise in the max starts to dominate. In the full run it is common to see plain DQN predict a $V(s_0)$ above the realized return, sometimes above the theoretical cap of 100, while Double DQN stays closer to the truth. Overestimation is not merely cosmetic: inflated values in some states distort the greedy policy in their neighbours, and on harder tasks (the Atari benchmark of the original paper) Double DQN's more accurate values translate into substantially better scores.

Two further improvements are standard and appear in the exercises: **prioritized experience replay** (Schaul et al., 2016) samples transitions with large TD error more often (Exercise 3), and the **dueling architecture** (Wang et al., 2016) splits the network into a state-value stream and an advantage stream (Exercise 5). Together with multi-step returns, distributional value estimates and noisy exploration layers they form the "Rainbow" agent of Hessel et al. (2018).

> **Common pitfall.** Evaluating the agent with the training $\varepsilon$. A policy that is nearly perfect under greedy actions can look mediocre if five percent of its actions are random, especially in tasks where a single wrong action ends the episode. Always report greedy (or otherwise deterministic) evaluation returns separately from training returns, as `evaluate_policy` does.

Notebook 13 · Reinforcement Learning — part 3 (actor-critic, GAE, PPO, RL in the wild, exercises, summary)

## 6. Actor-critic methods: A2C, generalized advantage estimation and PPO

### 6.1 Combining a policy with a learned critic

REINFORCE with a baseline already contained two networks: an **actor** (the policy) and a **critic** (the value function used as baseline). But it used the critic only to *shift* Monte Carlo returns, and it waited for the end of each episode to update. **Actor-critic** methods go further and let the critic *replace* the Monte Carlo return by bootstrapped estimates, exactly as TD learning replaced Monte Carlo evaluation in Section 3. The one-step version weights each action by the TD error

$$
\delta_t = r_{t+1} + \gamma V_\phi(s_{t+1}) - V_\phi(s_t),
$$

which is an estimate of the advantage $A^\pi(s_t, a_t)$: it compares the one-step-bootstrapped value of taking $a_t$ with the value of the state. Its variance is tiny compared to a full return (only one random reward and one random next state enter), but it is biased whenever $V_\phi$ is wrong, which is always the case early in training. Between the one-step TD error and the full Monte Carlo advantage lies a whole family of $n$-step estimators, and choosing $n$ is a bias–variance trade-off.

### 6.2 Generalized advantage estimation

Schulman et al. (2016) proposed to take an exponentially weighted average of *all* the $n$-step estimators, which has a beautifully simple form. Define the $n$-step advantage $\hat A^{(n)}_t = \sum_{l=0}^{n-1} \gamma^l r_{t+l+1} + \gamma^n V(s_{t+n}) - V(s_t)$; a short calculation shows that it telescopes into a sum of TD errors, $\hat A^{(n)}_t = \sum_{l=0}^{n-1} \gamma^l \delta_{t+l}$. Averaging the $n$-step estimators with weights $(1-\lambda)\lambda^{n-1}$ therefore gives the **generalized advantage estimator**

$$
\hat A^{\mathrm{GAE}(\gamma, \lambda)}_t \;=\; \sum_{l=0}^{\infty} (\gamma \lambda)^l\, \delta_{t+l},
\qquad\text{computed backwards as}\qquad
\hat A_t = \delta_t + \gamma \lambda\, \hat A_{t+1}.
$$

The parameter $\lambda \in [0, 1]$ interpolates between the two extremes: $\lambda = 0$ gives the one-step TD error (low variance, high bias) and $\lambda = 1$ gives the Monte Carlo return minus the baseline (unbiased, high variance). Values around $0.95$ are a robust default. The recursion has the same shape as the discounted-return recursion of Section 1, with $\gamma \lambda$ playing the role of the discount and $\delta_t$ the role of the reward. At the end of a fixed-length rollout the episode is usually not finished, so the last TD error bootstraps from $V(s_T)$ of the state the agent is in; at a terminal state the bootstrap is zero. The value targets for training the critic are $\hat R_t = \hat A_t + V(s_t)$ (the "$\lambda$-return").

Truncation by the time limit needs the same care as in DQN. When the environment reports `truncated` but not `terminated`, the state after the last step is not terminal, so we add $\gamma V(s_{\text{final}})$ to the last reward and then treat the step as the end of the sequence. This keeps the target unbiased without letting the advantage recursion run across the reset into the next episode.

### 6.3 Advantage actor-critic (A2C)

A2C (the synchronous version of A3C, Mnih et al., 2016) alternates between collecting a short rollout of $n$ steps (here $n = 32$) with the current policy and taking one gradient step on the combined loss

$$
L(\theta, \phi) = \underbrace{-\frac{1}{n}\sum_t \log \pi_\theta(a_t \mid s_t)\, \hat A_t}_{\text{policy loss}}
\;+\; c_v\, \underbrace{\frac{1}{n}\sum_t \big( V_\phi(s_t) - \hat R_t \big)^2}_{\text{value loss}}
\;-\; c_e\, \underbrace{\frac{1}{n}\sum_t \mathcal{H}\big( \pi_\theta(\cdot \mid s_t) \big)}_{\text{entropy bonus}} .
$$

The actor and the critic share the first layers of the network, so a single optimizer updates both; the coefficients $c_v$ and $c_e$ balance the three terms. We use $c_v = 0.25$ and $c_e = 0.01$: the value targets on CartPole are tens of units large, so with a shared trunk and gradient clipping a large $c_v$ lets the value loss dominate the clipped gradient and starves the policy (a real effect worth checking whenever an actor-critic "does not learn"). The **entropy bonus** $\mathcal{H}(\pi(\cdot \mid s)) = -\sum_a \pi(a \mid s) \log \pi(a \mid s)$ is maximal for the uniform policy and zero for a deterministic one; subtracting it from the loss rewards the policy for staying stochastic and prevents premature collapse onto a single action before the critic has learned anything, which is the policy-gradient analogue of the $\varepsilon$-greedy schedule. Exercise 4 studies its effect.

The code separates the rollout collection (a `Runner` that keeps the environment state between rollouts, so that episodes may span several updates) from the update itself, because PPO reuses the same runner.

In [ ]:
class ActorCritic(nn.Module):
    """Policy head (action logits) and value head (scalar) on top of a trunk that is shared (A2C) or separate (PPO).

    The policy head is initialized with a tiny gain so that the initial policy is almost uniform, a standard
    trick in reference implementations that keeps early updates small.
    """
    def __init__(self, obs_dim, n_actions, hidden=64, shared=True):
        super().__init__()
        def trunk():
            return nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh())
        self.shared = shared
        self.pi_body = trunk()
        self.v_body = self.pi_body if shared else trunk()
        self.pi_head = nn.Linear(hidden, n_actions)
        self.v_head = nn.Linear(hidden, 1)
        nn.init.orthogonal_(self.pi_head.weight, gain=0.01)
        nn.init.zeros_(self.pi_head.bias)

    def forward(self, x):
        h_pi = self.pi_body(x)
        h_v = h_pi if self.shared else self.v_body(x)
        return self.pi_head(h_pi), self.v_head(h_v).squeeze(-1)


def compute_gae(rewards, values, dones, last_value, gamma, lam):
    """Generalized advantage estimation over one rollout.

    dones[t] = 1 if the episode ended after step t (no bootstrap into step t+1); last_value = V(s_T).
    Returns (advantages, value targets).
    """
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    last_adv = 0.0
    for t in reversed(range(T)):
        next_value = last_value if t == T - 1 else values[t + 1]
        nonterminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * nonterminal - values[t]
        last_adv = delta + gamma * lam * nonterminal * last_adv
        adv[t] = last_adv
    return adv, adv + values


class Runner:
    """Collects fixed-length rollouts from one environment; episodes persist across rollouts."""
    def __init__(self, env, model, gamma, seed=SEED):
        self.env, self.model, self.gamma = env, model, gamma
        self.obs_dim = env.observation_space.shape[0]
        self.obs, _ = env.reset(seed=seed)
        self.ep_return, self.step = 0.0, 0
        self.ep_returns, self.steps_at = [], []

    def rollout(self, n_steps):
        obs_b = np.zeros((n_steps, self.obs_dim), np.float32)
        act_b = np.zeros(n_steps, np.int64)
        rew_b, done_b, val_b, logp_b = (np.zeros(n_steps, np.float32) for _ in range(4))
        for t in range(n_steps):
            obs_t = to_tensor(self.obs)
            with torch.no_grad():
                logits, v = self.model(obs_t)
                dist = Categorical(logits=logits)
                a = dist.sample()
                logp = dist.log_prob(a)
            action = int(a.item())
            next_obs, r, terminated, truncated, _ = self.env.step(action)
            self.step += 1
            self.ep_return += r
            obs_b[t], act_b[t], val_b[t], logp_b[t] = self.obs, action, v.item(), logp.item()
            rew_b[t], done_b[t] = r, float(terminated or truncated)
            if truncated and not terminated:                     # bootstrap through the time limit
                with torch.no_grad():
                    rew_b[t] += self.gamma * self.model(to_tensor(next_obs))[1].item()
            if terminated or truncated:
                self.ep_returns.append(self.ep_return); self.steps_at.append(self.step); self.ep_return = 0.0
                next_obs, _ = self.env.reset()
            self.obs = next_obs
        with torch.no_grad():
            last_value = self.model(to_tensor(self.obs))[1].item()
        return dict(obs=obs_b, actions=act_b, rewards=rew_b, dones=done_b, values=val_b, logp=logp_b,
                    last_value=last_value)

    def solved(self, threshold, n=20):
        return len(self.ep_returns) >= n and np.mean(self.ep_returns[-n:]) >= threshold


def a2c(env, total_steps, n_steps=32, gamma=0.99, lam=0.95, lr=2e-3, value_coef=0.25, ent_coef=0.01,
        max_grad_norm=1.0, solve_threshold=475.0, seed=SEED, log_every=2_000):
    """Advantage actor-critic with GAE and an entropy bonus, single environment, n-step rollouts.

    Returns (model, per-episode returns, env steps at the end of each episode, entropy after each update).
    """
    seed_everything(seed)
    model = ActorCritic(env.observation_space.shape[0], int(env.action_space.n)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    runner = Runner(env, model, gamma, seed)
    entropies, t0, next_log = [], time.time(), log_every
    while runner.step < total_steps:
        b = runner.rollout(n_steps)
        adv, ret = compute_gae(b["rewards"], b["values"], b["dones"], b["last_value"], gamma, lam)
        S, A = torch.as_tensor(b["obs"], device=device), torch.as_tensor(b["actions"], device=device)
        ADV, RET = torch.as_tensor(adv, device=device), torch.as_tensor(ret, device=device)
        logits, v = model(S)
        dist = Categorical(logits=logits)
        policy_loss = -(dist.log_prob(A) * ADV).mean()
        value_loss = F.mse_loss(v, RET)
        entropy = dist.entropy().mean()
        loss = policy_loss + value_coef * value_loss - ent_coef * entropy
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        opt.step()
        entropies.append(entropy.item())
        if runner.solved(solve_threshold):
            print(f"solved: mean return of the last 20 training episodes >= {solve_threshold:.0f} after {runner.step} steps")
            break
        if runner.step >= next_log:
            recent = np.mean(runner.ep_returns[-20:]) if runner.ep_returns else float("nan")
            print(f"step {runner.step:6d} | episodes {len(runner.ep_returns):4d} | mean return (last 20) {recent:6.1f} | "
                  f"entropy {entropy.item():.3f} | {time.time() - t0:5.1f}s")
            next_log += log_every
    return model, np.array(runner.ep_returns), np.array(runner.steps_at), np.array(entropies)

A2C_STEPS = 8_000 if FAST_RUN else 150_000
env = make_env("CartPole-v1")
print("--- A2C")
model_a2c, ret_a2c, steps_a2c, ent_a2c = a2c(env, A2C_STEPS, log_every=2_000 if FAST_RUN else 10_000)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(steps_a2c, ret_a2c, alpha=0.3); axes[0].plot(steps_a2c, moving_average(ret_a2c, 20))
axes[0].set_xlabel("environment steps"); axes[0].set_ylabel("return (moving average of 20)"); axes[0].set_title("A2C on CartPole")
axes[1].plot(np.arange(len(ent_a2c)) * 32, ent_a2c); axes[1].axhline(math.log(2), color="k", ls="--", label="log 2 (uniform policy)")
axes[1].set_xlabel("environment steps"); axes[1].set_ylabel("policy entropy"); axes[1].set_title("entropy of the policy"); axes[1].legend()
plt.tight_layout(); plt.show()

A2C updates the policy every 32 steps instead of once per episode, so it makes many more (smaller, noisier) updates per unit of experience than REINFORCE. The entropy plot shows the exploration story explicitly: the policy starts uniform (entropy $\log 2 \approx 0.69$ nats) and becomes more deterministic as the critic learns which actions are better; the entropy bonus keeps it from dropping to zero. Keep in mind that the return curve is that of the *stochastic* training policy, which by design still samples the worse action now and then; the deterministic policy that takes the arg-max can be far better, as the evaluation in Section 6.5 will show. If you see the return curve collapse after a good run, look at the entropy plot: a collapse to a deterministic policy that then meets a state it handles badly, with no exploration left to recover, is the most common failure mode of on-policy methods without an entropy term.

### 6.4 Proximal Policy Optimization

A2C takes exactly one gradient step per rollout and then throws the data away, because the policy gradient is only valid for samples from the *current* policy. It would be far more sample-efficient to take several steps on the same rollout, but after the first step the samples come from an older policy $\pi_{\theta_{\text{old}}}$. Importance sampling corrects for this: with the probability ratio

$$
\rho_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)},
$$

the surrogate $\mathbb{E}_{\theta_{\text{old}}}[\rho_t(\theta)\, \hat A_t]$ has the same gradient as the true objective at $\theta = \theta_{\text{old}}$, and remains a sensible objective nearby. The danger is that maximizing it aggressively drives $\rho_t$ far from one, into a region where the old samples say nothing about the new policy and the estimate becomes garbage. Trust-region methods constrain the step by a KL divergence; **PPO** (Schulman et al., 2017) achieves the same effect with a clipped objective that needs no constraint:

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_t\Big[ \min\!\Big( \rho_t(\theta)\, \hat A_t,\; \operatorname{clip}\big(\rho_t(\theta),\, 1 - \epsilon,\, 1 + \epsilon\big)\, \hat A_t \Big) \Big], \qquad \epsilon = 0.2 .
$$

Consider one term. If $\hat A_t > 0$ the objective wants to increase $\rho_t$; the clip stops rewarding increases beyond $1 + \epsilon$, so the gradient becomes zero once the action's probability has grown by 20%. If $\hat A_t < 0$ the objective wants to decrease $\rho_t$; the clip stops rewarding decreases below $1 - \epsilon$. The outer `min` makes the objective a *pessimistic lower bound*: the clipped value is used only when it is worse than the unclipped one, so a probability ratio that moved in the wrong direction is never protected by the clip. The net effect is that several epochs of mini-batch SGD on one rollout move the policy a bounded distance, and the algorithm is remarkably robust to hyperparameters. PPO also normalizes the advantages of each rollout to zero mean and unit variance (a per-batch rescaling that stabilizes the step size) and trains the value head with the same loss as A2C.

The implementation is compact: collect a rollout of `rollout_len` steps, compute GAE, store the old log-probabilities, and run `n_epochs` passes of mini-batch updates on the clipped loss. Three details follow the reference implementations (Spinning Up, CleanRL, Stable-Baselines3). The actor and the critic are *separate* networks, so that the large value-regression gradients cannot interfere with the policy through a shared trunk during the many updates per rollout. The learning rate is annealed linearly to zero over training, which makes the late updates small and the final policy stable. And the fraction of probability ratios that hit the clip is recorded as a diagnostic.

In [ ]:
def ppo(env, total_steps, rollout_len=512, n_epochs=4, minibatch_size=64, clip_eps=0.2, gamma=0.99, lam=0.95,
        lr=1e-3, value_coef=0.5, ent_coef=0.0, max_grad_norm=0.5, anneal_lr=True, solve_threshold=475.0,
        seed=SEED, log_every=4_096):
    """Single-environment PPO with the clipped surrogate objective, GAE, advantage normalization,
    separate actor and critic networks and linear learning-rate annealing.

    Returns (model, per-episode returns, env steps at the end of each episode, clip fraction per update).
    """
    seed_everything(seed)
    rng = np.random.default_rng(seed)
    model = ActorCritic(env.observation_space.shape[0], int(env.action_space.n), shared=False).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, eps=1e-5)
    runner = Runner(env, model, gamma, seed)
    clip_fracs, t0, next_log = [], time.time(), log_every
    while runner.step < total_steps:
        if anneal_lr:
            for group in opt.param_groups:
                group["lr"] = lr * max(0.0, 1.0 - runner.step / total_steps)
        b = runner.rollout(rollout_len)
        adv, ret = compute_gae(b["rewards"], b["values"], b["dones"], b["last_value"], gamma, lam)
        S, A = torch.as_tensor(b["obs"], device=device), torch.as_tensor(b["actions"], device=device)
        RET, OLD_LOGP = torch.as_tensor(ret, device=device), torch.as_tensor(b["logp"], device=device)
        ADV = torch.as_tensor(adv, device=device)
        ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)
        for epoch in range(n_epochs):
            perm = rng.permutation(rollout_len)
            for start in range(0, rollout_len, minibatch_size):
                idx = torch.as_tensor(perm[start:start + minibatch_size], device=device)
                logits, v = model(S[idx])
                dist = Categorical(logits=logits)
                ratio = torch.exp(dist.log_prob(A[idx]) - OLD_LOGP[idx])
                surr_unclipped = ratio * ADV[idx]
                surr_clipped = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * ADV[idx]
                policy_loss = -torch.min(surr_unclipped, surr_clipped).mean()
                value_loss = F.mse_loss(v, RET[idx])
                entropy = dist.entropy().mean()
                loss = policy_loss + value_coef * value_loss - ent_coef * entropy
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                opt.step()
                clip_fracs.append(((ratio - 1.0).abs() > clip_eps).float().mean().item())
        if runner.solved(solve_threshold):
            print(f"solved: mean return of the last 20 training episodes >= {solve_threshold:.0f} after {runner.step} steps")
            break
        if runner.step >= next_log:
            recent = np.mean(runner.ep_returns[-20:]) if runner.ep_returns else float("nan")
            print(f"step {runner.step:6d} | episodes {len(runner.ep_returns):4d} | mean return (last 20) {recent:6.1f} | "
                  f"clip fraction {np.mean(clip_fracs[-n_epochs * (rollout_len // minibatch_size):]):.3f} | {time.time() - t0:5.1f}s")
            next_log += log_every
    return model, np.array(runner.ep_returns), np.array(runner.steps_at), np.array(clip_fracs)

PPO_STEPS = 8_192 if FAST_RUN else 120_000
PPO_ROLLOUT = 256 if FAST_RUN else 512
print("--- PPO")
model_ppo, ret_ppo, steps_ppo, clip_ppo = ppo(env, PPO_STEPS, rollout_len=PPO_ROLLOUT,
                                              log_every=2_048 if FAST_RUN else 8_192)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(steps_ppo, ret_ppo, alpha=0.3); axes[0].plot(steps_ppo, moving_average(ret_ppo, 20))
axes[0].set_xlabel("environment steps"); axes[0].set_ylabel("return (moving average of 20)"); axes[0].set_title("PPO on CartPole")
axes[1].plot(moving_average(clip_ppo, 16)); axes[1].set_xlabel("gradient step"); axes[1].set_ylabel("fraction of clipped ratios")
axes[1].set_title("how often the clip is active")
plt.tight_layout(); plt.show()

PPO makes `n_epochs` passes over every rollout, so each environment step contributes to several gradient updates, and the clipped objective keeps those updates from wandering off. The clip fraction is typically a few percent to about twenty percent during the active phase of learning, and it decays towards zero at the end because the learning rate is annealed to zero; a fraction that stays near zero *early* in training means the policy barely moves per rollout (learning rate or epochs could be higher), while one that keeps growing means the updates are too aggressive. In the full run PPO usually reaches the solved threshold after a few tens of thousands of steps, which is the best sample efficiency of the on-policy methods in this notebook. This combination of sample reuse and stability is why PPO became the default algorithm for a huge range of applications, from continuous control and games to fine-tuning language models (Section 7).

### 6.5 All methods on one axis

We now put the learning curves of every deep method on a common axis of environment steps and evaluate each final policy deterministically (arg-max of the logits or Q-values). The budgets differ between methods, as the printed step counts show, so read the plot as "how much return per environment step" rather than as a race to the right edge; and remember that all curves come from a single seed, so differences of tens of return points are within run-to-run noise.

In [ ]:
def deterministic_act(net, actor_critic=False):
    """act(obs) -> arg-max action, for a policy network (logits) or an actor-critic (logits, value)."""
    def act(obs):
        with torch.no_grad():
            out = net(to_tensor(obs))
            logits = out[0] if actor_critic else out
        return int(logits.argmax().item())
    return act

pg_rets, pg_steps, pg_policy = pg_results["reward-to-go + baseline"]
curves = {"REINFORCE (reward-to-go + baseline)": (pg_steps, pg_rets), "DQN": (steps_dqn, ret_dqn),
          "Double DQN": (steps_ddqn, ret_ddqn), "A2C": (steps_a2c, ret_a2c), "PPO": (steps_ppo, ret_ppo)}
plt.figure(figsize=(8, 4.5))
for name, (steps, rets) in curves.items():
    plt.plot(steps, moving_average(rets, 20), label=name)
plt.axhline(475, color="k", ls=":", label="solved threshold (475)")
plt.xlabel("environment steps"); plt.ylabel("training return (moving average of 20)")
plt.title("Learning curves of all methods on CartPole (single seed)"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

final_policies = {"REINFORCE (reward-to-go + baseline)": deterministic_act(pg_policy),
                  "DQN": greedy_act(q_dqn), "Double DQN": greedy_act(q_ddqn),
                  "A2C": deterministic_act(model_a2c, actor_critic=True),
                  "PPO": deterministic_act(model_ppo, actor_critic=True)}
print(f"{'method':38s} {'env steps used':>15s} {'deterministic return (10 ep.)':>30s}")
for name, act in final_policies.items():
    mean_ret, std_ret = evaluate_policy(env, act, n_episodes=10)
    print(f"{name:38s} {int(curves[name][0][-1]):15d} {mean_ret:21.1f} +- {std_ret:5.1f}")

The picture that emerges is representative of the field even though the details vary between seeds. Monte Carlo policy gradients (REINFORCE) are simple and unbiased but need the most experience. Value-based methods with replay (DQN, Double DQN) reuse every transition many times and are the most sample-efficient on this small discrete-action task, at the price of more moving parts (buffer, target network, exploration schedule) and sensitivity to hyperparameters. On-policy actor-critic methods (A2C, PPO) sit in between: they are stable, easy to tune, and PPO's sample reuse narrows the gap to replay-based methods. In the full run all of them can reach the solved threshold; in `FAST_RUN` mode only the beginnings of their learning curves are visible, and the deterministic evaluation shows which of them already has a usable policy.

> **Going deeper.** Modern successors combine the ideas: Soft Actor-Critic (Haarnoja et al., 2018) is an off-policy actor-critic with replay and an entropy term in the *objective* rather than as a bonus; TD3 fixes overestimation for continuous actions with clipped double Q-learning; MuZero learns a model and plans with it. All of them are built from the components in this notebook: bootstrapped value targets, importance ratios, replay, target networks and entropy regularization.

## 7. Reinforcement learning in the wild, and RLHF

CartPole is solved in minutes on a laptop. Real problems are harder for reasons that have little to do with the algorithms above, and knowing them is as important as knowing the algorithms.

**Sample efficiency.** The agents in this notebook needed tens of thousands of interactions to balance a pole; Atari agents need tens of millions of frames, and robot-learning systems would wear out their hardware long before then. Most of the practical progress in RL is about getting more out of each sample: replay and off-policy learning, model-based methods, pretraining representations from unlabelled data, and learning in simulation. When each environment step is expensive (a physical robot, a chemistry experiment, a user of a recommender system), the sample count is the budget that dominates every decision.

**Reward design and reward hacking.** The reward function is the specification of the task, and RL will optimize exactly what is written, not what was meant. A boat-racing agent that discovered it could score more by circling in a lagoon collecting power-ups than by finishing the race, or a simulated robot that learned to exploit a physics-engine bug, are classic illustrations of **reward hacking**. Sparse rewards (only at the goal) are hard to learn from, so practitioners add *shaping* terms, and every shaping term is an opportunity to misspecify. Potential-based shaping $F(s, s') = \gamma \Phi(s') - \Phi(s)$ is the one form that provably leaves the optimal policy unchanged. This is a specification problem, not an optimization problem, and it connects directly to the alignment questions of Notebook 16 · Interpretability, Robustness and Uncertainty.

**Sim-to-real.** Training in simulation is cheap, but a policy trained in a simulator meets a different distribution in reality (friction, latencies, sensor noise, lighting). The standard remedy is **domain randomization**: randomize the simulator's parameters during training so that the policy becomes robust to the whole range and reality is just one more sample. Distribution shift, which we treat in Notebook 16, is thus built into the deployment of every RL system.

**Offline RL.** Often a fixed dataset of trajectories exists (hospital records, driving logs, past recommender interactions) but further exploration is unsafe or impossible. Off-policy algorithms such as DQN look applicable, but with no new data the deadly triad strikes with full force: the max in the target queries actions that were never observed, whose values the network extrapolates arbitrarily, and the policy converges on those hallucinated values. **Offline RL** methods (conservative Q-learning, implicit Q-learning and others, see Levine et al., 2020) constrain the learned policy to stay close to the actions present in the data.

**Model-based RL.** The tabular section showed that a known model makes planning trivial. Model-based methods learn $P$ and $R$ from data (a neural network predicting the next state) and then either plan with the model or generate imagined rollouts to train a policy. The pay-off is sample efficiency; the risk is that a policy will exploit the model's errors exactly as it exploits reward mis-specification. AlphaZero and MuZero showed that learning a model and searching with it can be state of the art in board games and Atari.

### 7.1 RLHF: reinforcement learning from human feedback

The most visible application of the ideas in this notebook is the last stage of training large language models, discussed from the modelling side in Notebook 10 · NLP with Pretrained Transformers (Hugging Face). Generating a response is a sequential decision problem: the state is the prompt plus the tokens generated so far, each action is the next token, and the policy is the language model itself. The difficulty is the reward, because there is no environment that scores a paragraph. RLHF (Christiano et al., 2017; Ouyang et al., 2022) builds one from human preferences in three steps.

1. **Supervised fine-tuning** produces a reasonable starting policy $\pi_{\text{ref}}$ from demonstrations.
2. **Reward modelling.** Humans compare pairs of responses $(y_w, y_l)$ to the same prompt $x$ and indicate which is better. A network $r_\phi(x, y)$ is trained so that the Bradley–Terry model of the comparison is likely: $\Pr(y_w \succ y_l \mid x) = \sigma\big( r_\phi(x, y_w) - r_\phi(x, y_l) \big)$, i.e. by minimizing $-\log \sigma\big( r_\phi(x, y_w) - r_\phi(x, y_l) \big)$ over the comparison dataset.
3. **RL fine-tuning.** The policy is trained with PPO to maximize the learned reward while staying close to the reference model, which is essential because the reward model is only accurate near the data it was trained on (reward hacking again):

$$
\max_\theta\; \mathbb{E}_{x \sim \mathcal{D},\, y \sim \pi_\theta(\cdot \mid x)} \Big[ r_\phi(x, y) \Big] \;-\; \beta\, \mathbb{E}_{x \sim \mathcal{D}} \Big[ \mathrm{KL}\big( \pi_\theta(\cdot \mid x)\, \|\, \pi_{\text{ref}}(\cdot \mid x) \big) \Big].
$$

In practice the KL term is added to the reward of each token as $-\beta \log \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{ref}}(a_t \mid s_t)}$, the sequence-level reward $r_\phi$ arrives at the last token, a value head estimates $V(s_t)$ for GAE, and the clipped PPO objective of Section 6.4 is optimized over mini-batches of generated responses. Every component of that pipeline, the reward-to-go, the baseline, the entropy considerations, the probability ratio and its clip, is one we implemented on CartPole.

The KL-regularized objective has a closed-form optimum, $\pi^*(y \mid x) \propto \pi_{\text{ref}}(y \mid x) \exp\big( r(x, y) / \beta \big)$, which lets one write the reward as a function of the policy and train it directly on the comparison data. This is **Direct Preference Optimization** (Rafailov et al., 2023), whose loss appears and is implemented on a toy batch in Notebook 10; it removes the reward model and the RL loop at the cost of some flexibility. Both routes are in active use, and hybrids that use RL with verifiable rewards (unit tests for code, checkable answers for mathematics) rather than learned reward models are increasingly common. Whichever variant one uses, the vocabulary is that of this notebook: policies, rewards, advantages, and the ever-present risk that an optimizer will find what the reward *says* rather than what it *means*.

## Exercises

**Exercise 1 (warm-up).** Compare SARSA and Q-learning on the *non-slippery* FrozenLake (`is_slippery=False`) and on the slippery one, with the same training budget. Plot the learning curves for both environments, print the greedy success rate of each learned policy, and explain in one paragraph why the two methods agree on the deterministic lake and (usually) disagree on the slippery one.

**Solution to Exercise 1**

In [ ]:
N_EX1 = 2_000 if FAST_RUN else 6_000
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ex1_policies = {}
for ax, slippery in zip(axes, [False, True]):
    env_ex1 = make_env("FrozenLake-v1", is_slippery=slippery)
    for method in ["q_learning", "sarsa"]:
        Q_e, rets_e = tabular_td(env_ex1, N_EX1, method=method)
        pol = Q_e.argmax(axis=1)
        ex1_policies[(slippery, method)] = pol
        succ = evaluate_policy(env_ex1, lambda s, pol=pol: int(pol[s]), n_episodes=500)[0]
        ax.plot(moving_average(rets_e, 100), label=f"{method} (greedy success {succ:.2f})")
    ax.set_title(f"FrozenLake, is_slippery={slippery}"); ax.set_xlabel("training episode")
    ax.set_ylabel("success rate (moving average of 100)"); ax.legend()
plt.tight_layout(); plt.show()

for slippery in [False, True]:
    same = np.array_equal(ex1_policies[(slippery, "q_learning")], ex1_policies[(slippery, "sarsa")])
    n_diff = int((ex1_policies[(slippery, "q_learning")] != ex1_policies[(slippery, "sarsa")]).sum())
    print(f"is_slippery={slippery}: greedy policies identical = {same} ({n_diff} of 16 states differ)")

On the deterministic lake both methods learn quickly and their greedy policies reach the goal every time: the shortest safe path is six steps long and, once the value of the goal has propagated back, there is no reason for the two update rules to disagree on which action is best in the states that matter. (They may still differ in states off the learned path, where the values are nearly equal and the tie is broken by noise; the state count printed above shows this.) On the slippery lake SARSA evaluates the $\varepsilon$-greedy policy, so it accounts for the fact that five percent of its own actions are random and, near holes, prefers actions that stay safe even when the *next* action is random. Q-learning evaluates the greedy policy and learns the value of acting perfectly from now on. When exploration is switched off for evaluation, Q-learning's policy is the one that was optimized for exactly that situation, which is why it usually scores higher; SARSA's would be the better choice if the agent had to keep exploring while acting, as an on-line system often does.

**Exercise 2.** Exploration schedules. Using `tabular_td` on the slippery lake, compare four $\varepsilon$ schedules with the same budget: a constant $\varepsilon = 0.1$, a constant $\varepsilon = 0.5$, the linear decay used in the text, and an exponential decay $\varepsilon_k = \max(0.05,\, e^{-k / \tau})$ with $\tau$ equal to one fifth of the number of episodes. Plot the schedules and the learning curves, and report the greedy success rate of each final policy. Which schedules learn the best *final* policy, and which have the best *training* returns? Why can the two rankings differ?

**Solution to Exercise 2**

In [ ]:
N_EX2 = 3_000 if FAST_RUN else 8_000
tau = N_EX2 / 5
schedules = {"constant 0.1": lambda ep: 0.1,
             "constant 0.5": lambda ep: 0.5,
             "linear 1.0 -> 0.05": linear_eps(N_EX2),
             "exponential": lambda ep: max(0.05, math.exp(-ep / tau))}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
episodes = np.arange(N_EX2)
for name, fn in schedules.items():
    axes[0].plot(episodes, [fn(e) for e in episodes], label=name)
axes[0].set_xlabel("episode"); axes[0].set_ylabel("epsilon"); axes[0].set_title("exploration schedules"); axes[0].legend()

print(f"{'schedule':22s} {'train success (last 500)':>26s} {'greedy success (1000 ep.)':>27s}")
for name, fn in schedules.items():
    Q_e, rets_e = tabular_td(lake, N_EX2, method="q_learning", eps_fn=fn)
    pol = Q_e.argmax(axis=1)
    greedy = evaluate_policy(lake, lambda s, pol=pol: int(pol[s]), n_episodes=1_000)[0]
    print(f"{name:22s} {rets_e[-500:].mean():26.3f} {greedy:27.3f}")
    axes[1].plot(moving_average(rets_e, 200), label=name)
axes[1].set_xlabel("episode"); axes[1].set_ylabel("training success rate (MA 200)"); axes[1].set_title("Q-learning with each schedule"); axes[1].legend()
plt.tight_layout(); plt.show()

The training curves rank the schedules by how greedy they are late in training: a constant $\varepsilon = 0.5$ acts randomly half of the time and therefore has poor training returns *even if its Q-table is good*, whereas the decaying schedules and $\varepsilon = 0.1$ exploit what they know. The greedy evaluation tells a different story: heavy exploration visits every state–action pair many times and produces an accurate table, so the greedy policy extracted from the $\varepsilon = 0.5$ run is often as good as any. The exponential schedule illustrates the opposite risk. It is already below $0.15$ after $2\tau$ episodes (40% of the budget) and at its floor after about $3\tau$; it explores less in total than the linear schedule and spends most of that exploration in the first few hundred episodes, when the table is still empty, so from then on the agent mostly revisits the states its early, half-formed policy prefers; the greedy policy extracted from it can be clearly worse than the others even though its *training* returns look fine. This is the general lesson for off-policy methods: training returns measure the behaviour policy, the greedy evaluation measures what was learned, and only the second is the goal. A schedule that decays over a substantial fraction of the budget is the practical compromise, because it gives both: broad exploration early, when the table is uninformative anyway, and useful behaviour later.

**Exercise 3.** Prioritized experience replay (simple version). Implement a `PrioritizedReplayBuffer` that samples transition $i$ with probability $p_i^\alpha / \sum_j p_j^\alpha$, where the priority $p_i = |\delta_i| + \epsilon$ is the magnitude of the transition's last TD error (new transitions get the current maximum priority), and returns importance-sampling weights $w_i = (N\,P(i))^{-\beta}$, normalized by their maximum, where $P(i)$ is the sampling probability and $N$ the number of stored transitions; they correct the bias of non-uniform sampling. Plug it into `dqn` via `buffer_cls` and compare with the uniform Double DQN run of Section 5.3. A linear-time `rng.choice` over the priorities is fine here; the original paper uses a sum tree for $O(\log N)$ sampling.

**Solution to Exercise 3**

In [ ]:
class PrioritizedReplayBuffer(ReplayBuffer):
    """Proportional prioritized replay (Schaul et al., 2016) with O(N) sampling; fine for small buffers."""
    def __init__(self, capacity, obs_dim, alpha=0.6, beta=0.4, eps=1e-3):
        super().__init__(capacity, obs_dim)
        self.priorities = np.zeros(capacity, np.float32)
        self.alpha, self.beta, self.eps = alpha, beta, eps

    def add(self, obs, action, reward, next_obs, done):
        self.priorities[self.ptr] = self.priorities[:self.size].max() if self.size > 0 else 1.0
        super().add(obs, action, reward, next_obs, done)

    def sample(self, batch_size, rng):
        p = self.priorities[:self.size] ** self.alpha
        p = p / p.sum()
        idx = rng.choice(self.size, size=batch_size, p=p)
        weights = (self.size * p[idx]) ** (-self.beta)
        weights = weights / weights.max()
        batch = (torch.as_tensor(self.obs[idx], device=device),
                 torch.as_tensor(self.actions[idx], device=device),
                 torch.as_tensor(self.rewards[idx], device=device),
                 torch.as_tensor(self.next_obs[idx], device=device),
                 torch.as_tensor(self.dones[idx], device=device))
        return batch, idx, torch.as_tensor(weights, dtype=torch.float32, device=device)

    def update_priorities(self, idx, td_errors):
        self.priorities[idx] = np.abs(td_errors) + self.eps

print("--- Double DQN with prioritized replay")
q_per, ret_per, steps_per, _ = dqn(env, DQN_STEPS, double=True, eps_decay_steps=DQN_EPS_DECAY,
                                   buffer_cls=PrioritizedReplayBuffer, log_every=DQN_LOG)
plt.figure()
plt.plot(steps_ddqn, moving_average(ret_ddqn, 20), label="Double DQN, uniform replay")
plt.plot(steps_per, moving_average(ret_per, 20), label="Double DQN, prioritized replay")
plt.xlabel("environment steps"); plt.ylabel("return (moving average of 20)"); plt.title("Prioritized vs uniform replay")
plt.legend(); plt.tight_layout(); plt.show()
for name, q_net in [("uniform", q_ddqn), ("prioritized", q_per)]:
    mean_ret, std_ret = evaluate_policy(env, greedy_act(q_net), n_episodes=10)
    print(f"{name:12s}: deterministic return {mean_ret:6.1f} +- {std_ret:5.1f}")

Prioritized replay focuses the updates on transitions the network predicts badly, which are typically the rare and informative ones (the last steps before a fall, the first successful recoveries). The importance weights $w_i$ shrink the gradient of over-sampled transitions so that the expected update still matches the uniform loss; $\beta$ is often annealed from 0.4 to 1 over training so that the correction is exact by the end, which we omitted for brevity. On CartPole the gain is modest and can be hidden by seed noise; on the Atari benchmark the paper reports a large improvement. The cost is the sampling itself: our O(N) `rng.choice` is fine for 50,000 transitions but a sum tree is needed for buffers of a million.

**Exercise 4.** Entropy regularization ablation. Train A2C with entropy coefficients $c_e \in \{0, 0.01, 0.1\}$ and the same budget. Plot the learning curves and the policy entropy over training for each run, and describe the effect of too little and too much regularization.

**Solution to Exercise 4**

In [ ]:
A2C_STEPS_EX4 = 5_000 if FAST_RUN else 40_000
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ent_coef in [0.0, 0.01, 0.1]:
    print(f"--- A2C with ent_coef = {ent_coef}")
    _, rets_e, steps_e, ents_e = a2c(env, A2C_STEPS_EX4, ent_coef=ent_coef, log_every=10 ** 9)
    axes[0].plot(steps_e, moving_average(rets_e, 20), label=f"ent_coef = {ent_coef}")
    axes[1].plot(np.arange(len(ents_e)) * 32, moving_average(ents_e, 10), label=f"ent_coef = {ent_coef}")
    print(f"    final mean return (last 20 episodes): {np.mean(rets_e[-20:]):.1f}, final entropy {ents_e[-1]:.3f}")
axes[0].set_xlabel("environment steps"); axes[0].set_ylabel("return (moving average of 20)"); axes[0].set_title("A2C returns"); axes[0].legend()
axes[1].axhline(math.log(2), color="k", ls="--"); axes[1].set_xlabel("environment steps"); axes[1].set_ylabel("policy entropy (MA 10)")
axes[1].set_title("policy entropy"); axes[1].legend()
plt.tight_layout(); plt.show()

Without the bonus ($c_e = 0$) the entropy tends to fall quickly, and the run is at the mercy of whichever action the early gradient steps favoured: sometimes it learns fine, sometimes it commits to a poor policy and cannot recover because it no longer tries anything else. With $c_e = 0.1$ the bonus dominates the small advantages early in training, the policy stays close to uniform (entropy near $\log 2$) and the agent behaves almost randomly for a long time; on CartPole the maximum entropy bonus per step (0.069) is comparable to the policy loss, so learning is slowed noticeably. The intermediate value $c_e = 0.01$ is the usual default: enough to prevent collapse, small enough that the return term drives the policy once the advantages become informative. The right value is task-dependent and scales with the reward magnitude, which is one reason reward normalization is common in practice.

**Exercise 5 (challenge).** Dueling DQN. Implement the dueling architecture of Wang et al. (2016): after a shared layer, one stream outputs the state value $V(s)$ and the other the advantages $A(s, a)$, combined as $Q(s, a) = V(s) + A(s, a) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s, a')$. Explain why the mean-subtraction is necessary for identifiability, plug the network into `dqn` via `net_cls` (with `double=True`), compare with the plain Double DQN, and verify that the network works on both a single observation and a batch.

**Solution to Exercise 5**

In [ ]:
class DuelingQNetwork(nn.Module):
    """Dueling architecture: Q(s, a) = V(s) + A(s, a) - mean_a' A(s, a')."""
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU())
        self.value_stream = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.adv_stream = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_actions))
    def forward(self, x):
        h = self.body(x)
        adv = self.adv_stream(h)
        return self.value_stream(h) + adv - adv.mean(dim=-1, keepdim=True)

# shape check: a single observation and a batch
duel_test = DuelingQNetwork(4, 2).to(device)
print("single obs ->", tuple(duel_test(torch.zeros(4, device=device)).shape),
      "| batch of 8 ->", tuple(duel_test(torch.zeros(8, 4, device=device)).shape))

print("--- Dueling Double DQN")
q_duel, ret_duel, steps_duel, _ = dqn(env, DQN_STEPS, double=True, eps_decay_steps=DQN_EPS_DECAY,
                                      net_cls=DuelingQNetwork, log_every=DQN_LOG)
plt.figure()
plt.plot(steps_ddqn, moving_average(ret_ddqn, 20), label="Double DQN")
plt.plot(steps_duel, moving_average(ret_duel, 20), label="Dueling Double DQN")
plt.xlabel("environment steps"); plt.ylabel("return (moving average of 20)"); plt.title("Dueling architecture")
plt.legend(); plt.tight_layout(); plt.show()
for name, q_net in [("Double DQN", q_ddqn), ("Dueling Double DQN", q_duel)]:
    mean_ret, std_ret = evaluate_policy(env, greedy_act(q_net), n_episodes=10)
    print(f"{name:20s}: deterministic return {mean_ret:6.1f} +- {std_ret:5.1f}")

The decomposition $Q = V + A$ is not unique: adding a constant to $V$ and subtracting it from every $A(s, a)$ leaves $Q$ unchanged, so without a constraint the two streams cannot be identified and training is ill-conditioned. Forcing the advantages to have zero mean over actions removes the degree of freedom (the paper notes that subtracting the *max* would be the more principled choice, making $V$ equal to the value of the greedy action, but that the mean is more stable in practice). The benefit of the architecture is that the value stream is updated by *every* transition through $V(s)$, whereas in a plain Q-network only the output of the taken action receives a gradient; in states where the action hardly matters (most CartPole states with the pole near vertical), this shares information across actions and speeds up learning. Like the other refinements, the improvement on CartPole is modest and seed-dependent; it becomes significant in problems with many actions.

## Summary

- An MDP is defined by states, actions, a transition kernel, a reward function and a discount factor; the Markov property lets us write value functions of the current state only, and the discount factor sets an effective horizon of about $1/(1-\gamma)$ steps.
- The Bellman expectation equation is linear in $V^\pi$ and can be solved exactly or by iterating a $\gamma$-contraction; the Bellman optimality equation replaces the policy average by a max and is solved by value iteration when the model is known.
- Q-learning and SARSA replace the expectation over next states by sampled transitions. Q-learning is off-policy and learns $Q^*$ regardless of the behaviour policy; SARSA is on-policy and learns the value of the exploring policy it follows.
- The policy gradient theorem turns the gradient of the expected return into an expectation of $\nabla \log \pi$ weighted by returns, with the unknown dynamics dropping out. Reward-to-go and state-dependent baselines reduce the variance of the estimator without introducing bias.
- DQN makes Q-learning with neural networks work through experience replay (decorrelated, reusable data), a target network (stationary regression targets), an $\varepsilon$ schedule and the Huber loss. Double DQN removes most of the overestimation caused by the max in the target by decoupling action selection from evaluation.
- Truncation by a time limit is not termination: the correct handling is to keep bootstrapping through it, both in the replay buffer and in advantage estimation.
- Actor-critic methods replace Monte Carlo returns by bootstrapped advantages; GAE interpolates between TD and Monte Carlo estimates with a single parameter $\lambda$, and an entropy bonus keeps the policy exploring.
- PPO's clipped surrogate objective allows several epochs of updates on one rollout while keeping the policy within a trust region, which makes it robust, sample-efficient and the default algorithm for many applications, including RLHF for language models.
- Learning curves in RL are noisy and seed-dependent: smooth them, compare on the axis of environment steps, evaluate deterministically, and run several seeds before drawing conclusions.
- Real deployments are dominated by sample efficiency, reward specification and distribution shift; RL optimizes exactly the reward that is written down, so reward hacking must be anticipated rather than discovered.

## Further reading

- Sutton, R. S. and Barto, A. G., 2018, *Reinforcement Learning: An Introduction* (2nd edition), MIT Press. The standard textbook; Chapters 3–6 cover MDPs, dynamic programming, Monte Carlo and TD methods, Chapter 13 policy gradients. Freely available at http://incompleteideas.net/book/the-book-2nd.html
- Williams, R. J., 1992, Simple Statistical Gradient-Following Algorithms for Connectionist Reinforcement Learning, *Machine Learning* 8. The original REINFORCE paper.
- Sutton, R. S., McAllester, D., Singh, S. and Mansour, Y., 2000, Policy Gradient Methods for Reinforcement Learning with Function Approximation, *NeurIPS*.
- Mnih, V. et al., 2015, Human-level Control through Deep Reinforcement Learning, *Nature* 518. The DQN paper.
- van Hasselt, H., Guez, A. and Silver, D., 2016, Deep Reinforcement Learning with Double Q-learning, *AAAI*.
- Schaul, T., Quan, J., Antonoglou, I. and Silver, D., 2016, Prioritized Experience Replay, *ICLR*; Wang, Z. et al., 2016, Dueling Network Architectures for Deep Reinforcement Learning, *ICML*; Hessel, M. et al., 2018, Rainbow: Combining Improvements in Deep Reinforcement Learning, *AAAI*.
- Mnih, V. et al., 2016, Asynchronous Methods for Deep Reinforcement Learning, *ICML*. Introduces A3C, of which A2C is the synchronous variant.
- Schulman, J., Moritz, P., Levine, S., Jordan, M. and Abbeel, P., 2016, High-Dimensional Continuous Control Using Generalized Advantage Estimation, *ICLR*.
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A. and Klimov, O., 2017, Proximal Policy Optimization Algorithms, arXiv:1707.06347.
- Christiano, P. et al., 2017, Deep Reinforcement Learning from Human Preferences, *NeurIPS*; Ouyang, L. et al., 2022, Training Language Models to Follow Instructions with Human Feedback, *NeurIPS*; Rafailov, R. et al., 2023, Direct Preference Optimization: Your Language Model is Secretly a Reward Model, *NeurIPS*.
- Levine, S., Kumar, A., Tucker, G. and Fu, J., 2020, Offline Reinforcement Learning: Tutorial, Review, and Perspectives on Open Problems, arXiv:2005.01643.
- Gymnasium documentation: https://gymnasium.farama.org/ and OpenAI Spinning Up in Deep RL: https://spinningup.openai.com/ (clear derivations and reference implementations of VPG, PPO, DDPG, TD3 and SAC).